# Titan Experiment Runner
This notebook sets up the environment and runs the Associative Recall experiment on the Titans model.

In [1]:
# Install dependencies
!pip install torch deepspeed transformers tqdm einops sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 30.6 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.0/54.0 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 20.0 MB/s eta 0:00:00
  Created wheel for deepspeed: filename=deepspeed-0.18.4-py3-none-any.whl size=1772943 sha256=c35144980021d8135170f698feb2d21322d33b472efa85b63e4c3ad634bdb73a
  Stored in directory: /root/.cache/pip/wheels/cc/92/85/b2a464b7d12a2d0e895296b3e4759d3a86f3a3dc959918d043
Successfully built deepspeed


In [2]:
import os
import base64

print(f'Current working directory: {os.getcwd()}')

def write_file(path, b64_content):
    target_path = os.path.abspath(path)
    directory = os.path.dirname(target_path)
    if directory:
        os.makedirs(directory, exist_ok=True)
    with open(target_path, "wb") as f:
        f.write(base64.b64decode(b64_content))
    print(f"Created {target_path}")

files = {
    'main.py': '...',
    'projects/original_benchmarks/associative_recall.py': 'CmltcG9ydCB0b3JjaApmcm9tIHRvcmNoLnV0aWxzLmRhdGEgaW1wb3J0IERhdGFzZXQKaW1wb3J0IHJhbmRvbQoKY2xhc3MgQXNzb2NpYXRpdmVSZWNhbGxEYXRhc2V0KERhdGFzZXQpOgogICAgZGVmIF9faW5pdF9fKHNlbGYsIHZvY2FiX3NpemU9MTAwLCBzZXFfbGVuPTEyOCwgbnVtX3NhbXBsZXM9MTAwMDAsIHBhaXJfbGVuPTIpOgogICAgICAgICIiIgogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIHZvY2FiX3NpemU6IFNpemUgb2YgdGhlIHZvY2FidWxhcnkgKGtleXMgYW5kIHZhbHVlcykuCiAgICAgICAgICAgIHNlcV9sZW46IFRvdGFsIGxlbmd0aCBvZiB0aGUgc2VxdWVuY2UuCiAgICAgICAgICAgIG51bV9zYW1wbGVzOiBOdW1iZXIgb2Ygc2FtcGxlcyBpbiB0aGUgZGF0YXNldC4KICAgICAgICAgICAgcGFpcl9sZW46IE51bWJlciBvZiB0b2tlbnMgcGVyIGtleS12YWx1ZSBwYWlyICh1c3VhbGx5IDI6IGtleSwgdmFsdWUpLgogICAgICAgICIiIgogICAgICAgIHNlbGYudm9jYWJfc2l6ZSA9IHZvY2FiX3NpemUKICAgICAgICBzZWxmLnNlcV9sZW4gPSBzZXFfbGVuCiAgICAgICAgc2VsZi5udW1fc2FtcGxlcyA9IG51bV9zYW1wbGVzCiAgICAgICAgc2VsZi5wYWlyX2xlbiA9IHBhaXJfbGVuCiAgICAgICAgCiAgICAgICAgIyBSZXNlcnZlIGV4cGxpY2l0IHRva2VucwogICAgICAgIHNlbGYuUVVFUllfVE9LRU4gPSB2b2NhYl9zaXplIC0gMSAgIyBOb3Qgc3RyaWN0bHkgbmVlZGVkIGlmIHBvc2l0aW9uIGltcGxpY2l0bHkgZGVmaW5lcyBxdWVyeSwgYnV0IGdvb2QgZm9yIGNsYXJpdHkgaWYgd2UgdXNlZCBzcGVjaWFsIHRva2VucwogICAgICAgICMgRm9yIHNpbXBsaWNpdHkgaW4gdGhpcyBudW1lcmljIHRhc2ssIHdlIGp1c3QgdXNlIHRoZSBudW1iZXJzIFswLCB2b2NhYl9zaXplLTJdIGZvciBkYXRhCiAgICAgICAgCiAgICBkZWYgX19sZW5fXyhzZWxmKToKICAgICAgICByZXR1cm4gc2VsZi5udW1fc2FtcGxlcwoKICAgIGRlZiBfX2dldGl0ZW1fXyhzZWxmLCBpZHgpOgogICAgICAgICMgMS4gR2VuZXJhdGUgdW5pcXVlIGtleXMKICAgICAgICAjIFdlIG5lZWQgZW5vdWdoIGtleXMgdG8gZmlsbCBtb3N0IG9mIHRoZSBzZXF1ZW5jZS4KICAgICAgICAjIE1heCBwYWlycyBhcHByb3ggc2VxX2xlbiAvIDIKICAgICAgICBudW1fcGFpcnMgPSAoc2VsZi5zZXFfbGVuIC0gMSkgLy8gMiAKICAgICAgICAKICAgICAgICAjIEVuc3VyZSB2b2NhYiBzaXplIGlzIHN1ZmZpY2llbnQgZm9yIHVuaXF1ZSBrZXlzIGlmIHBvc3NpYmxlLCBvciBhbGxvdyByZXBlYXRzIGlmIHZvY2FiIGlzIHNtYWxsIChidXQgQVIgdXN1YWxseSByZXF1aXJlcyB1bmlxdWUga2V5cyBmb3IgcXVlcnlpbmcpCiAgICAgICAgIyBMZXQncyBhc3N1bWUgdm9jYWJfc2l6ZSA+PiBudW1fcGFpcnMgdXN1YWxseSwgb3Igd2Ugc2FtcGxlIHdpdGggcmVwbGFjZW1lbnQgYnV0IHRoYXQgbWFrZXMgdGFzayBoYXJkZXIvYW1iaWd1b3VzLgogICAgICAgICMgU3RhbmRhcmQgQVI6IGtleXMgYXJlIHVuaXF1ZS4KICAgICAgICAKICAgICAgICBhdmFpbGFibGVfa2V5cyA9IGxpc3QocmFuZ2Uoc2VsZi52b2NhYl9zaXplIC0gMTApKSAjIFJlc2VydmUgc29tZSB0b3AgdG9rZW5zIGZvciBzcGVjaWFsIHVzZSBpZiBuZWVkZWQKICAgICAgICBpZiBsZW4oYXZhaWxhYmxlX2tleXMpIDwgbnVtX3BhaXJzOgogICAgICAgICAgICAjIElmIHZvY2FiIGlzIHNtYWxsLCB3ZSByZWR1Y2UgbnVtX3BhaXJzCiAgICAgICAgICAgIG51bV9wYWlycyA9IGxlbihhdmFpbGFibGVfa2V5cykKICAgICAgICAgICAgCiAgICAgICAga2V5cyA9IHJhbmRvbS5zYW1wbGUoYXZhaWxhYmxlX2tleXMsIG51bV9wYWlycykKICAgICAgICB2YWx1ZXMgPSBbcmFuZG9tLmNob2ljZShhdmFpbGFibGVfa2V5cykgZm9yIF8gaW4gcmFuZ2UobnVtX3BhaXJzKV0KICAgICAgICAKICAgICAgICAjIDIuIFNlbGVjdCBhIHRhcmdldCBxdWVyeQogICAgICAgIHRhcmdldF9pZHggPSByYW5kb20ucmFuZGludCgwLCBudW1fcGFpcnMgLSAxKQogICAgICAgIHF1ZXJ5X2tleSA9IGtleXNbdGFyZ2V0X2lkeF0KICAgICAgICB0YXJnZXRfdmFsID0gdmFsdWVzW3RhcmdldF9pZHhdCiAgICAgICAgCiAgICAgICAgIyAzLiBDb25zdHJ1Y3Qgc2VxdWVuY2UKICAgICAgICAjIGlucHV0OiBrMSB2MSBrMiB2MiAuLi4ga04gdk4gcXVlcnlfa2V5CiAgICAgICAgIyBsYWJlbDogLTEgLTEgLTEgLTEgLi4uIC0xIC0xIHRhcmdldF92YWwKICAgICAgICAKICAgICAgICBpbnB1dF9pZHMgPSBbXQogICAgICAgIGxhYmVscyA9IFtdCiAgICAgICAgCiAgICAgICAgZm9yIGssIHYgaW4gemlwKGtleXMsIHZhbHVlcyk6CiAgICAgICAgICAgIGlucHV0X2lkcy5leHRlbmQoW2ssIHZdKQogICAgICAgICAgICBsYWJlbHMuZXh0ZW5kKFstMTAwLCAtMTAwXSkgIyBJZ25vcmUgbG9zcyBmb3IgY29udGV4dAogICAgICAgICAgICAKICAgICAgICAjIEFwcGVuZCBxdWVyeQogICAgICAgIGlucHV0X2lkcy5hcHBlbmQocXVlcnlfa2V5KQogICAgICAgIGxhYmVscy5hcHBlbmQodGFyZ2V0X3ZhbCkKICAgICAgICAKICAgICAgICAjIFBhZGRpbmcgaWYgbmVlZGVkIChzaW1wbGlmaWVkOiBqdXN0IHRydW5jYXRlIG9yIGFzc3VtZSBmaXhlZCBzaXplIGxvZ2ljIG1hdGNoZXMpCiAgICAgICAgIyBGb3IgdGhpcyBkZW1vLCB3ZSByZXR1cm4gdGVuc29ycyBkaXJlY3RseS4KICAgICAgICAjIElmIHRoZSBnZW5lcmF0ZWQgbGVuZ3RoIDwgc2VsZi5zZXFfbGVuLCB3ZSBtaWdodCBwYWQgKGJ1dCBiYXRjaGluZyBoYW5kbGVzIGl0IGlmIHdlIHVzZSBjb2xsYXRlKS4KICAgICAgICAjIExldCdzIHNpbXBsZSByZXR1cm4gYXMgaXMsIGFuZCB0aGUgbW9kZWwgaGFuZGxlcyB2YXJpYWJsZSBsZW5ndGggb3Igd2UgbWFrZSBpdCBmaXhlZC4KICAgICAgICAKICAgICAgICByZXR1cm4gdG9yY2gudGVuc29yKGlucHV0X2lkcywgZHR5cGU9dG9yY2gubG9uZyksIHRvcmNoLnRlbnNvcihsYWJlbHMsIGR0eXBlPXRvcmNoLmxvbmcpCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgZHMgPSBBc3NvY2lhdGl2ZVJlY2FsbERhdGFzZXQoKQogICAgeCwgeSA9IGRzWzBdCiAgICBwcmludCgiSW5wdXQ6IiwgeCkKICAgIHByaW50KCJMYWJlbDoiLCB5KQo=',
    'projects/original_benchmarks/train_toy.py': 'CmltcG9ydCBzeXMKaW1wb3J0IG9zCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IGFyZ3BhcnNlCmZyb20gdG9yY2gudXRpbHMuZGF0YSBpbXBvcnQgRGF0YUxvYWRlcgpmcm9tIHRxZG0gaW1wb3J0IHRxZG0KCiMgQWRkIHBhcmVudCBkaXJlY3RvcnkgdG8gcGF0aCB0byBpbXBvcnQgbWFpbiAob3JpZ2luYWxfYmVuY2htYXJrcyAtPiBwcm9qZWN0cyAtPiBUaXRhbkxMTSkKc3lzLnBhdGguYXBwZW5kKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKSkpCmZyb20gbWFpbiBpbXBvcnQgVGl0YW5NQUMsIFRpdGFuTUFHLCBUaXRhbk1BTCwgVGl0YW5Nb2RlbEZvckxNCgpmcm9tIGFzc29jaWF0aXZlX3JlY2FsbCBpbXBvcnQgQXNzb2NpYXRpdmVSZWNhbGxEYXRhc2V0CgpkZWYgdHJhaW4oYXJncyk6CiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKICAgIHByaW50KGYiVXNpbmcgZGV2aWNlOiB7ZGV2aWNlfSIpCgogICAgIyBEYXRhc2V0CiAgICB0cmFpbl9kcyA9IEFzc29jaWF0aXZlUmVjYWxsRGF0YXNldCh2b2NhYl9zaXplPWFyZ3Mudm9jYWJfc2l6ZSwgc2VxX2xlbj1hcmdzLnNlcV9sZW4sIG51bV9zYW1wbGVzPTEwMDAwKQogICAgdHJhaW5fZGwgPSBEYXRhTG9hZGVyKHRyYWluX2RzLCBiYXRjaF9zaXplPWFyZ3MuYmF0Y2hfc2l6ZSwgc2h1ZmZsZT1UcnVlKQogICAgCiAgICAjIE1vZGVsIFNldHVwCiAgICBpZiBhcmdzLm1vZGVsX3ZhcmlhbnQgPT0gIk1BQyI6CiAgICAgICAgdGl0YW5fbW9kdWxlID0gVGl0YW5NQUMoYXJncy5kaW0sIGFyZ3MuY2h1bmtfc2l6ZSwgYXJncy5oaWRkZW5fZGltLCBhcmdzLm1lbW9yeV9kZXB0aCwgYXJncy5udW1fcGVyc2lzdGVudF90b2tlbnMsIHRocmVzaG9sZD1hcmdzLnRocmVzaG9sZCkKICAgIGVsaWYgYXJncy5tb2RlbF92YXJpYW50ID09ICJNQUciOgogICAgICAgIHRpdGFuX21vZHVsZSA9IFRpdGFuTUFHKGFyZ3MuZGltLCBhcmdzLmhpZGRlbl9kaW0sIGFyZ3MubWVtb3J5X2RlcHRoLCBhcmdzLm51bV9wZXJzaXN0ZW50X3Rva2VucywgYXJncy53aW5kb3dfc2l6ZSwgdGhyZXNob2xkPWFyZ3MudGhyZXNob2xkKQogICAgZWxzZTogIyBNQUwKICAgICAgICB0aXRhbl9tb2R1bGUgPSBUaXRhbk1BTChhcmdzLmRpbSwgYXJncy5oaWRkZW5fZGltLCBhcmdzLm1lbW9yeV9kZXB0aCwgYXJncy5udW1fcGVyc2lzdGVudF90b2tlbnMsIGFyZ3Mud2luZG93X3NpemUsIHRocmVzaG9sZD1hcmdzLnRocmVzaG9sZCkKICAgICAgICAKICAgIG1vZGVsID0gVGl0YW5Nb2RlbEZvckxNKHRpdGFuX21vZHVsZSwgYXJncy52b2NhYl9zaXplLCBhcmdzLmRpbSkKICAgIG1vZGVsLnRvKGRldmljZSkKICAgIAogICAgb3B0aW1pemVyID0gdG9yY2gub3B0aW0uQWRhbVcobW9kZWwucGFyYW1ldGVycygpLCBscj0xZS00KQogICAgCiAgICBtb2RlbC50cmFpbigpCiAgICAKICAgIGZvciBlcG9jaCBpbiByYW5nZShhcmdzLm51bV9lcG9jaHMpOgogICAgICAgIHRvdGFsX2xvc3MgPSAwCiAgICAgICAgY29ycmVjdCA9IDAKICAgICAgICB0b3RhbCA9IDAKICAgICAgICAKICAgICAgICBwYmFyID0gdHFkbSh0cmFpbl9kbCwgZGVzYz1mIkVwb2NoIHtlcG9jaCsxfSIpCiAgICAgICAgZm9yIGlucHV0X2lkcywgbGFiZWxzIGluIHBiYXI6CiAgICAgICAgICAgIGlucHV0X2lkcywgbGFiZWxzID0gaW5wdXRfaWRzLnRvKGRldmljZSksIGxhYmVscy50byhkZXZpY2UpCiAgICAgICAgICAgIAogICAgICAgICAgICBvcHRpbWl6ZXIuemVyb19ncmFkKCkKICAgICAgICAgICAgbG9naXRzLCBsb3NzID0gbW9kZWwoaW5wdXRfaWRzLCBsYWJlbHMpICMgVGl0YW5Nb2RlbEZvckxNIGhhbmRsZXMgbG9zcyBjYWxjdWxhdGlvbiBpZiBsYWJlbHMgcHJvdmlkZWQKICAgICAgICAgICAgCiAgICAgICAgICAgIGxvc3MuYmFja3dhcmQoKQogICAgICAgICAgICBvcHRpbWl6ZXIuc3RlcCgpCiAgICAgICAgICAgIAogICAgICAgICAgICB0b3RhbF9sb3NzICs9IGxvc3MuaXRlbSgpCiAgICAgICAgICAgIAogICAgICAgICAgICAjIEFjY3VyYWN5IE1ldHJpYwogICAgICAgICAgICAjIGxhYmVscyBoYXZlIC0xMDAgZm9yIGlnbm9yZWQgcG9zaXRpb25zLiBXZSBvbmx5IGNhcmUgYWJvdXQgdGhlIGxhc3QgdG9rZW4gdXN1YWxseS4KICAgICAgICAgICAgIyBGbGF0dGVuIGZvciBhcmdtYXgKICAgICAgICAgICAgcHJlZF90b2tlbnMgPSB0b3JjaC5hcmdtYXgobG9naXRzLCBkaW09LTEpCiAgICAgICAgICAgIG1hc2sgPSBsYWJlbHMgIT0gLTEwMAogICAgICAgICAgICAKICAgICAgICAgICAgY29ycmVjdCArPSAocHJlZF90b2tlbnNbbWFza10gPT0gbGFiZWxzW21hc2tdKS5zdW0oKS5pdGVtKCkKICAgICAgICAgICAgdG90YWwgKz0gbWFzay5zdW0oKS5pdGVtKCkKICAgICAgICAgICAgCiAgICAgICAgICAgIHBiYXIuc2V0X3Bvc3RmaXgoeyJMb3NzIjogZiJ7bG9zcy5pdGVtKCk6LjRmfSIsICJBY2MiOiBmIntjb3JyZWN0L3RvdGFsOi4yJX0ifSkKICAgICAgICAgICAgCiAgICBwcmludCgiVHJhaW5pbmcgZmluaXNoZWQuIikKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBwYXJzZXIgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1vZGVsX3ZhcmlhbnQiLCB0eXBlPXN0ciwgZGVmYXVsdD0iTUFDIiwgY2hvaWNlcz1bIk1BQyIsICJNQUciLCAiTUFMIl0pCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXZvY2FiX3NpemUiLCB0eXBlPWludCwgZGVmYXVsdD0xMjgpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXNlcV9sZW4iLCB0eXBlPWludCwgZGVmYXVsdD02NCkKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZGltIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTI4KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1oaWRkZW5fZGltIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTI4KQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1tZW1vcnlfZGVwdGgiLCB0eXBlPWludCwgZGVmYXVsdD0yKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1udW1fcGVyc2lzdGVudF90b2tlbnMiLCB0eXBlPWludCwgZGVmYXVsdD0yKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1jaHVua19zaXplIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MzIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXdpbmRvd19zaXplIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MzIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWJhdGNoX3NpemUiLCB0eXBlPWludCwgZGVmYXVsdD0xNikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tbnVtX2Vwb2NocyIsIHR5cGU9aW50LCBkZWZhdWx0PTUpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLXRocmVzaG9sZCIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4wKQogICAgCiAgICBhcmdzID0gcGFyc2VyLnBhcnNlX2FyZ3MoKQogICAgdHJhaW4oYXJncykK',
    'projects/hybrid_titans/rag_compare.py': 'aW1wb3J0IHRvcmNoCmltcG9ydCBhcmdwYXJzZQppbXBvcnQgcmFuZG9tCmltcG9ydCBzeXMKaW1wb3J0IG9zCgojIEFkZCBwYXJlbnQgZGlyZWN0b3J5IHRvIHBhdGggdG8gaW1wb3J0IG1haW4gKGh5YnJpZF90aXRhbnMgLT4gcHJvamVjdHMgLT4gVGl0YW5MTE0pCnN5cy5wYXRoLmFwcGVuZChvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmFic3BhdGgoX19maWxlX18pKSkpKQoKZnJvbSBtYWluIGltcG9ydCBUaXRhbk1BQywgVGl0YW5NQUcsIFRpdGFuTUFMLCBUaXRhblJBRwoKZGVmIGdlbmVyYXRlX2hheXN0YWNrKG51bV9kb2NzLCBkb2NfbGVuLCBkaW0sIGtleV9pZHg9Tm9uZSk6CiAgICAiIiIKICAgIEdlbmVyYXRlcyBOIGRvY3VtZW50cy4gT25lIGNvbnRhaW5pbmcgdGhlICduZWVkbGUnIChwYXNza2V5KSwgb3RoZXJzIG5vaXNlLgogICAgIiIiCiAgICBkb2NzID0gW10KICAgIG5lZWRsZV9wYXR0ZXJuID0gdG9yY2gucmFuZG4oMSwgZGltKQogICAgCiAgICBpZiBrZXlfaWR4IGlzIE5vbmU6CiAgICAgICAga2V5X2lkeCA9IHJhbmRvbS5yYW5kaW50KDAsIG51bV9kb2NzIC0gMSkKICAgICAgICAKICAgIGZvciBpIGluIHJhbmdlKG51bV9kb2NzKToKICAgICAgICBkb2MgPSB0b3JjaC5yYW5kbihkb2NfbGVuLCBkaW0pCiAgICAgICAgaWYgaSA9PSBrZXlfaWR4OgogICAgICAgICAgICBkb2NbLTEsIDpdID0gbmVlZGxlX3BhdHRlcm4KICAgICAgICBkb2NzLmFwcGVuZChkb2MpCiAgICAgICAgCiAgICByZXR1cm4gZG9jcywgbmVlZGxlX3BhdHRlcm4sIGtleV9pZHgKCmRlZiBwcmV0cmFpbl9pZGVudGl0eShtb2RlbCwgZGltLCBzdGVwcz01MDApOgogICAgIiIiCiAgICBUcmFpbnMgdGhlIG1vZGVsJ3MgQXR0ZW50aW9uL1Byb2Nlc3NpbmcgbGF5ZXJzIHRvIHBlcmZvcm0gSWRlbnRpdHkgTWFwcGluZyBmKHgpPXguCiAgICAiIiIKICAgIG9wdGltaXplciA9IHRvcmNoLm9wdGltLkFkYW1XKG1vZGVsLnBhcmFtZXRlcnMoKSwgbHI9MC4wMSkKICAgIAogICAgcHJpbnQoZiIgIFtQcmUtdHJhaW5pbmddIFRlYWNoaW5nIG1vZGVsIHRvIGNvcHkgKElkZW50aXR5IEZ1bmN0aW9uKS4uLiIpCiAgICAKICAgICMgW0ZpeF0gRGlzYWJsZSBUaXRhbidzIGludGVybmFsIFRlc3QtVGltZSBUcmFpbmluZyBkdXJpbmcgSWRlbnRpdHkgUHJlLXRyYWluaW5nLgogICAgIyBXZSBvbmx5IHdhbnQgdG8gdHJhaW4gdGhlIEF0dGVudGlvbiBtZWNoYW5pc20gKFdlaWdodHMpLCBub3QgdGhlIE1lbW9yeSAoU3RhdGUpLgogICAgIyBPdGhlcndpc2UsIGluLXBsYWNlIHVwZGF0ZXMgY2F1c2UgJ2JhY2t3YXJkIHRocm91Z2ggZ3JhcGggc2Vjb25kIHRpbWUnIGVycm9yLgogICAgc2F2ZWRfdGhyZXNob2xkID0gbW9kZWwudGhyZXNob2xkCiAgICBtb2RlbC50aHJlc2hvbGQgPSBmbG9hdCgnaW5mJykgIyBGb3JjZSBza2lwIHVwZGF0ZQogICAgCiAgICBmb3IgaSBpbiByYW5nZShzdGVwcyk6CiAgICAgICAgIyBHZW5lcmF0ZSByYW5kb20gZGF0YQogICAgICAgIHggPSB0b3JjaC5yYW5kbigxLCAzMiwgZGltKSAjIFtCYXRjaCwgU2VxLCBEaW1dCiAgICAgICAgdGFyZ2V0ID0geAogICAgICAgIAogICAgICAgIG9wdGltaXplci56ZXJvX2dyYWQoKQogICAgICAgIAogICAgICAgIHByZWQgPSBtb2RlbCh4KQogICAgICAgIGxvc3MgPSAoKHByZWQgLSB0YXJnZXQpKioyKS5tZWFuKCkKICAgICAgICBsb3NzLmJhY2t3YXJkKCkKICAgICAgICBvcHRpbWl6ZXIuc3RlcCgpCiAgICAgICAgCiAgICAgICAgaWYgaSAlIDIwID09IDA6CiAgICAgICAgICAgIHByaW50KGYiICAgIFN0ZXAge2l9OiBMb3NzIHtsb3NzLml0ZW0oKTouNGZ9IikKICAgICAgICAgICAgCiAgICAjIFJlc3RvcmUgb3JpZ2luYWwgdGhyZXNob2xkIGZvciBFeHBlcmltZW50CiAgICBtb2RlbC50aHJlc2hvbGQgPSBzYXZlZF90aHJlc2hvbGQKCmRlZiBydW5fc2luZ2xlX3ZhcmlhbnQodmFyaWFudF9uYW1lLCBNb2RlbENsYXNzLCBhcmdzLCBkb2NzX2JhdGNoLCBrZXlfaWR4KToKICAgIHByaW50KGYiXG5be3ZhcmlhbnRfbmFtZX1dIFRlc3RpbmcgTWVtb3J5IFJldGVudGlvbi4uLiIpCiAgICAKICAgICMgMS4gQ29uZmlnIEhhbmRsaW5nCiAgICAjIE1BQyB1c2VzICdjaHVua19zaXplJywgTUFHL01BTCB1c2UgJ3dpbmRvd19zaXplJwogICAgY29uZmlnID0gewogICAgICAgICJkaW0iOiBhcmdzLmRpbSwKICAgICAgICAiaGlkZGVuX2RpbSI6IGFyZ3MuZGltLAogICAgICAgICJtZW1vcnlfZGVwdGgiOiAyLAogICAgICAgICJudW1fcGVyc2lzdGVudF90b2tlbnMiOiAwLAogICAgICAgICJ0aHJlc2hvbGQiOiAwLjAKICAgIH0KICAgIAogICAgaWYgdmFyaWFudF9uYW1lID09ICdUaXRhbk1BQyc6CiAgICAgICAgY29uZmlnWyJjaHVua19zaXplIl0gPSAzMgogICAgZWxzZToKICAgICAgICBjb25maWdbIndpbmRvd19zaXplIl0gPSAzMgogICAgICAgIAogICAgdHJ5OgogICAgICAgIG1vZGVsX2Jhc2UgPSBNb2RlbENsYXNzKCoqY29uZmlnKQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgIHByaW50KGYiICBFcnJvciBpbnN0YW50aWF0aW5nIHt2YXJpYW50X25hbWV9OiB7ZX0iKQogICAgICAgIHJldHVybgoKICAgICMgW0NSSVRJQ0FMIFNURVBdIFByZS10cmFpbiB0aGUgJ0NvbnRleHQgRW5naW5lJyAoQXR0ZW50aW9uKQogICAgIyBPdGhlcndpc2UgSHlicmlkIFJBRyBmYWlscyBiZWNhdXNlIFJhbmRvbSBXZWlnaHRzIGRvbid0IENvcHkuCiAgICBwcmV0cmFpbl9pZGVudGl0eShtb2RlbF9iYXNlLCBhcmdzLmRpbSkKCiAgICAjIFtPcHRpbWl6YXRpb24gMi4wXSBIaWdoLUZpZGVsaXR5IFR1bmluZwogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgbW9kZWxfYmFzZS5sdG0udGhldGEuZmlsbF8oMC4wMDEpIAogICAgICAgIG1vZGVsX2Jhc2UubHRtLmFscGhhLmZpbGxfKDAuMDAwMSkKICAgIAogICAgbW9kZWxfcmFnID0gVGl0YW5SQUcobW9kZWxfYmFzZSkKICAgIAogICAgIyAyLiBCYXNlbGluZSBFdmFsdWF0aW9uCiAgICBuZWVkbGVfZG9jID0gZG9jc19iYXRjaFtrZXlfaWR4XQogICAgdGFyZ2V0ID0gbmVlZGxlX2RvYwogICAgcHJlZF9pbml0ID0gbW9kZWxfcmFnLnRpdGFuLmx0bS5mb3J3YXJkX25vX3VwZGF0ZSh0YXJnZXQucmVzaGFwZSgtMSwgYXJncy5kaW0pKQogICAgZGlmZl9pbml0ID0gKHByZWRfaW5pdCAtIHRhcmdldC5yZXNoYXBlKC0xLCBhcmdzLmRpbSkpCiAgICBsb3NzX2luaXQgPSAwLjUqKGRpZmZfaW5pdCoqMikuc3VtKCkuaXRlbSgpCiAgICAKICAgICMgMy4gRGlnZXN0aW9uCiAgICAjIHByaW50KGYiICBEaWdlc3RpbmcgKDIwMCBlcG9jaHMgcGVyIGRvYykuLi4iKQogICAgZm9yIGksIGRvYyBpbiBlbnVtZXJhdGUoZG9jc19iYXRjaCk6CiAgICAgICAgZm9yIF8gaW4gcmFuZ2UoMjAwKTogCiAgICAgICAgICAgIG1vZGVsX3JhZy5kaWdlc3Rfa25vd2xlZGdlKGRvYykKICAgICAgICAgICAgCiAgICAjIDQuIEZpbmFsIEV2YWx1YXRpb24KICAgIHByZWRfZmluYWwgPSBtb2RlbF9yYWcudGl0YW4ubHRtLmZvcndhcmRfbm9fdXBkYXRlKHRhcmdldC5yZXNoYXBlKC0xLCBhcmdzLmRpbSkpCiAgICBkaWZmX2ZpbmFsID0gKHByZWRfZmluYWwgLSB0YXJnZXQucmVzaGFwZSgtMSwgYXJncy5kaW0pKQogICAgbG9zc19hZnRlciA9IDAuNSooZGlmZl9maW5hbCoqMikuc3VtKCkuaXRlbSgpCiAgICAKICAgICMgNS4gTWV0cmljcwogICAgdG90YWxfZWxlbWVudHMgPSBhcmdzLmRvY19sZW4gKiBhcmdzLmRpbQogICAgc2lnbmFsX2VuZXJneSA9ICgodGFyZ2V0LnJlc2hhcGUoLTEsIGFyZ3MuZGltKSkqKjIpLnN1bSgpLml0ZW0oKQogICAgZXJyb3JfZW5lcmd5ID0gMiAqIGxvc3NfYWZ0ZXIKICAgIAogICAgZmlkZWxpdHkgPSAxLjAgLSAoZXJyb3JfZW5lcmd5IC8gKHNpZ25hbF9lbmVyZ3kgKyAxZS05KSkKICAgIGZpZGVsaXR5ID0gbWF4KDAuMCwgZmlkZWxpdHkpICogMTAwLjAKICAgIAogICAgcHJpbnQoZiIgIEluaXRpYWwgTG9zczoge2xvc3NfaW5pdDouNGZ9IikKICAgIHByaW50KGYiICBGaW5hbCBMb3NzOiAgIHtsb3NzX2FmdGVyOi40Zn0iKQogICAgcHJpbnQoZiIgIEZpZGVsaXR5OiAgICAge2ZpZGVsaXR5Oi4yZn0lIikKICAgIAogICAgaWYgbG9zc19hZnRlciA8IGxvc3NfaW5pdDoKICAgICAgICBwcmludChmIiAgUmVzdWx0OiBQQVNTIChJbXByb3ZlbWVudCB7KChsb3NzX2luaXQtbG9zc19hZnRlcikvbG9zc19pbml0KSoxMDA6LjFmfSUpIikKICAgIGVsc2U6CiAgICAgICAgcHJpbnQoZiIgIFJlc3VsdDogRkFJTCIpCiAgICAgICAgCiAgICAjIC0tLSBNZXRob2QgNDogSHlicmlkIFJBRyAoVGhlIEJlc3Qgb2YgQm90aCBXb3JsZHMpIC0tLQogICAgcHJpbnQoZiJcblt7dmFyaWFudF9uYW1lfV0gVGVzdGluZyBIeWJyaWQgUkFHIChXZWlnaHQgKyBDb250ZXh0KS4uLiIpCiAgICAjIFNjZW5hcmlvOiBOZWVkbGUgaXMgaW4gQ29udGV4dCwgTm9pc2UgaXMgaW4gV2VpZ2h0cy4KICAgIAogICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgIyBJbnB1dDogTmVlZGxlIERvYyAoQmF0Y2g9MSwgTGVuPWRvY19sZW4sIERpbT1kaW0pCiAgICAgICAgIyBCeSBwYXNzaW5nIGl0IGFzIGlucHV0ICd4JyB0byB0aGUgbW9kZWwgZm9yd2FyZCBwYXNzLCAKICAgICAgICAjIHRoZSBBdHRlbnRpb24gbWVjaGFuaXNtIHNlZXMgaXQgZGlyZWN0bHkuCiAgICAgICAgaW5wID0gbmVlZGxlX2RvYyAjIEFscmVhZHkgaGFzIGJhdGNoIGRpbSBbMSwgMzIsIDY0XQogICAgICAgIG91dHB1dCA9IG1vZGVsX3JhZyhpbnApCiAgICAgICAgCiAgICBkaWZmX2h5YnJpZCA9IChvdXRwdXQgLSBpbnApCiAgICBsb3NzX2h5YnJpZCA9IDAuNSooZGlmZl9oeWJyaWQqKjIpLnN1bSgpLml0ZW0oKQogICAgCiAgICBlcnJvcl9lbmVyZ3lfaHliID0gMiAqIGxvc3NfaHlicmlkCiAgICBmaWRlbGl0eV9oeWIgPSAxLjAgLSAoZXJyb3JfZW5lcmd5X2h5YiAvIChzaWduYWxfZW5lcmd5ICsgMWUtOSkpCiAgICBmaWRlbGl0eV9oeWIgPSBtYXgoMC4wLCBmaWRlbGl0eV9oeWIpICogMTAwLjAKICAgIAogICAgcHJpbnQoZiIgIEh5YnJpZCBGaWRlbGl0eToge2ZpZGVsaXR5X2h5YjouMmZ9JSAoU2hvdWxkIGJlIH4xMDAlKSIpCiAgICBpZiBmaWRlbGl0eV9oeWIgPiA5OS4wOgogICAgICAgIHByaW50KGYiICBSZXN1bHQ6IFBBU1MgKFBlcmZlY3QgUmVjYWxsIHZpYSBBdHRlbnRpb24pIikKCmRlZiBjb21wYXJlX21ldGhvZHMoYXJncyk6CiAgICBwcmludCgiLS0tIENvbmR1Y3RpbmcgUkFHIENvbXBhcmlzb24gRXhwZXJpbWVudCAoQWxsIFZhcmlhbnRzKSAtLS0iKQogICAgcHJpbnQoZiJEb2NzOiB7YXJncy5udW1fZG9jc30sIERvYyBMZW46IHthcmdzLmRvY19sZW59LCBEaW06IHthcmdzLmRpbX0iKQogICAgCiAgICAjIDEuIFNldHVwIERhdGEKICAgIGRvY3MsIF8sIGtleV9pZHggPSBnZW5lcmF0ZV9oYXlzdGFjayhhcmdzLm51bV9kb2NzLCBhcmdzLmRvY19sZW4sIGFyZ3MuZGltKQogICAgZG9jc19iYXRjaCA9IFtkLnVuc3F1ZWV6ZSgwKSBmb3IgZCBpbiBkb2NzXQogICAgCiAgICAjIDIuIFRyYWRpdGlvbmFsIFJBRyBCYXNlbGluZSAoT25seSBmb3IgTUFDLCBhcyBjb250ZXh0IGNvbmNhdCBpcyB1bmlxdWUgdG8gaXQgY29uY2VwdHVhbGx5KQogICAgcHJpbnQoIlxuW0Jhc2VsaW5lOiBUcmFkaXRpb25hbCBSQUcgKENvbmNhdGVuYXRpb24pXSIpCiAgICAjIE9ubHkgTUFDIGlzIHJlYWxseSBzdWl0YWJsZSBmb3IgJ0NvbmNhdCBhbGwnIG5haXZlIFJBRywgb3RoZXJzIHVzZSBzbGlkaW5nIHdpbmRvdy9nYXRpbmcuCiAgICAjIEJ1dCBmb3IgT09NIHRlc3RpbmcsIGFueSBtb2RlbCByZWNlaXZpbmcgaHVnZSBpbnB1dCB3b3Jrcy4KICAgIHRyeToKICAgICAgICBmdWxsX2NvbnRleHQgPSB0b3JjaC5jYXQoZG9jc19iYXRjaCwgZGltPTEpCiAgICAgICAgaWYgZnVsbF9jb250ZXh0LnNpemUoMSkgPiAyMDQ4OgogICAgICAgICAgICBwcmludChmIiAgUmVzdWx0OiBGQUlMIChDb250ZXh0IExlbmd0aCB7ZnVsbF9jb250ZXh0LnNpemUoMSl9ID4gMjA0OCkiKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHByaW50KGYiICBSZXN1bHQ6IFBBU1MgKFByb2Nlc3NlZCB7ZnVsbF9jb250ZXh0LnNpemUoMSl9IHRva2VucykiKQogICAgZXhjZXB0IEV4Y2VwdGlvbiBhcyBlOgogICAgICAgICBwcmludChmIiAgUmVzdWx0OiBGQUlMICh7ZX0pIikKCiAgICAjIDMuIFRpdGFuUkFHIFZhcmlhbnRzCiAgICB2YXJpYW50cyA9IFsKICAgICAgICAoJ1RpdGFuTUFDJywgVGl0YW5NQUMpLAogICAgICAgICgnVGl0YW5NQUcnLCBUaXRhbk1BRyksCiAgICAgICAgKCdUaXRhbk1BTCcsIFRpdGFuTUFMKQogICAgXQogICAgCiAgICBmb3IgbmFtZSwgY2xzIGluIHZhcmlhbnRzOgogICAgICAgIHJ1bl9zaW5nbGVfdmFyaWFudChuYW1lLCBjbHMsIGFyZ3MsIGRvY3NfYmF0Y2gsIGtleV9pZHgpCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1udW1fZG9jcyIsIHR5cGU9aW50LCBkZWZhdWx0PTUwLCBoZWxwPSJOdW1iZXIgb2YgcmV0cmlldmVkIGRvY3VtZW50cyIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWRvY19sZW4iLCB0eXBlPWludCwgZGVmYXVsdD02NCwgaGVscD0iTGVuZ3RoIG9mIGVhY2ggZG9jdW1lbnQiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kaW0iLCB0eXBlPWludCwgZGVmYXVsdD02NCkKICAgIGFyZ3MgPSBwYXJzZXIucGFyc2VfYXJncygpCiAgICAKICAgIGNvbXBhcmVfbWV0aG9kcyhhcmdzKQo=',
    'projects/hybrid_titans/sample_essays.py': 'IiIiClNhbXBsZSBFc3NheXMgZm9yIFRpdGFuUkFHIEV4cGVyaW1lbnRzCgpUaGVzZSBlc3NheXMgY29udGFpbiBmYWN0dWFsIGluZm9ybWF0aW9uIHRoYXQgY2FuIGJlIHVzZWQgdG8gdGVzdAp0aGUgbW9kZWwncyBhYmlsaXR5IHRvIG1lbW9yaXplIGFuZCByZWNhbGwgc3BlY2lmaWMgZGV0YWlscy4KIiIiCgpFU1NBWV9DTElNQVRFID0gIiIiCiMgVGhlIFNjaWVuY2Ugb2YgQ2xpbWF0ZSBDaGFuZ2UKCiMjIEludHJvZHVjdGlvbgpDbGltYXRlIGNoYW5nZSByZWZlcnMgdG8gbG9uZy10ZXJtIHNoaWZ0cyBpbiBnbG9iYWwgdGVtcGVyYXR1cmVzIGFuZCB3ZWF0aGVyIHBhdHRlcm5zLiAKV2hpbGUgbmF0dXJhbCBmYWN0b3JzIGxpa2Ugdm9sY2FuaWMgZXJ1cHRpb25zIGFuZCBzb2xhciBjeWNsZXMgY2FuIGluZmx1ZW5jZSBjbGltYXRlLCAKaHVtYW4gYWN0aXZpdGllcyBoYXZlIGJlZW4gdGhlIHByaW1hcnkgZHJpdmVyIHNpbmNlIHRoZSAxODAwcy4KCiMjIEtleSBGYWN0cwotIFRoZSBnbG9iYWwgYXZlcmFnZSB0ZW1wZXJhdHVyZSBoYXMgcmlzZW4gYnkgYXBwcm94aW1hdGVseSAxLjHCsEMgc2luY2UgcHJlLWluZHVzdHJpYWwgdGltZXMuCi0gVGhlIGNvbmNlbnRyYXRpb24gb2YgQ08yIGluIHRoZSBhdG1vc3BoZXJlIHJlYWNoZWQgNDIxIHBhcnRzIHBlciBtaWxsaW9uIChwcG0pIGluIDIwMjMuCi0gU2VhIGxldmVscyBoYXZlIHJpc2VuIGJ5IGFib3V0IDIwIGNlbnRpbWV0ZXJzICg4IGluY2hlcykgc2luY2UgMTkwMC4KLSBUaGUgQXJjdGljIGlzIHdhcm1pbmcgbmVhcmx5IGZvdXIgdGltZXMgZmFzdGVyIHRoYW4gdGhlIGdsb2JhbCBhdmVyYWdlLgoKIyMgQ2F1c2VzClRoZSBidXJuaW5nIG9mIGZvc3NpbCBmdWVsc+KAlGNvYWwsIG9pbCwgYW5kIG5hdHVyYWwgZ2Fz4oCUcmVsZWFzZXMgY2FyYm9uIGRpb3hpZGUgKENPMikgCmFuZCBvdGhlciBncmVlbmhvdXNlIGdhc2VzIGludG8gdGhlIGF0bW9zcGhlcmUuIFRoZXNlIGdhc2VzIHRyYXAgaGVhdCBmcm9tIHRoZSBzdW4sIApjcmVhdGluZyBhICJncmVlbmhvdXNlIGVmZmVjdCIgdGhhdCB3YXJtcyB0aGUgcGxhbmV0LgoKIyMgSW1wYWN0cwoxLiAqKkV4dHJlbWUgV2VhdGhlcioqOiBNb3JlIGZyZXF1ZW50IGhlYXR3YXZlcywgZHJvdWdodHMsIGFuZCBpbnRlbnNlIHN0b3Jtcy4KMi4gKipFY29zeXN0ZW0gRGlzcnVwdGlvbioqOiBDb3JhbCBibGVhY2hpbmcsIHNwZWNpZXMgbWlncmF0aW9uLCBhbmQgYmlvZGl2ZXJzaXR5IGxvc3MuCjMuICoqSHVtYW4gSGVhbHRoKio6IEluY3JlYXNlZCBoZWF0LXJlbGF0ZWQgaWxsbmVzc2VzIGFuZCBzcHJlYWQgb2YgdmVjdG9yLWJvcm5lIGRpc2Vhc2VzLgo0LiAqKkFncmljdWx0dXJlKio6IFNoaWZ0cyBpbiBncm93aW5nIHNlYXNvbnMgYW5kIHJlZHVjZWQgY3JvcCB5aWVsZHMgaW4gc29tZSByZWdpb25zLgoKIyMgU29sdXRpb25zCi0gVHJhbnNpdGlvbiB0byByZW5ld2FibGUgZW5lcmd5IHNvdXJjZXMgKHNvbGFyLCB3aW5kLCBoeWRybykuCi0gSW1wcm92ZSBlbmVyZ3kgZWZmaWNpZW5jeSBpbiBidWlsZGluZ3MgYW5kIHRyYW5zcG9ydGF0aW9uLgotIFByb3RlY3QgYW5kIHJlc3RvcmUgZm9yZXN0cywgd2hpY2ggYWJzb3JiIENPMi4KLSBJbXBsZW1lbnQgY2FyYm9uIHByaWNpbmcgdG8gaW5jZW50aXZpemUgZW1pc3Npb24gcmVkdWN0aW9ucy4KCiMjIENvbmNsdXNpb24KQWRkcmVzc2luZyBjbGltYXRlIGNoYW5nZSByZXF1aXJlcyB1cmdlbnQsIGNvb3JkaW5hdGVkIGdsb2JhbCBhY3Rpb24uIFRoZSBQYXJpcyBBZ3JlZW1lbnQgCm9mIDIwMTUgc2V0IHRoZSBnb2FsIG9mIGxpbWl0aW5nIHdhcm1pbmcgdG8gMS41wrBDIGFib3ZlIHByZS1pbmR1c3RyaWFsIGxldmVscy4KIiIiCgpFU1NBWV9BSSA9ICIiIgojIFRoZSBFdm9sdXRpb24gb2YgQXJ0aWZpY2lhbCBJbnRlbGxpZ2VuY2UKCiMjIEhpc3RvcmljYWwgQmFja2dyb3VuZApUaGUgdGVybSAiQXJ0aWZpY2lhbCBJbnRlbGxpZ2VuY2UiIHdhcyBjb2luZWQgYnkgSm9obiBNY0NhcnRoeSBpbiAxOTU2IGF0IHRoZSBEYXJ0bW91dGggCkNvbmZlcmVuY2UuIEVhcmx5IEFJIHJlc2VhcmNoIGZvY3VzZWQgb24gc3ltYm9saWMgcmVhc29uaW5nIGFuZCBwcm9ibGVtLXNvbHZpbmcuCgojIyBUaW1lbGluZSBvZiBNYWpvciBNaWxlc3RvbmVzCi0gKioxOTUwKio6IEFsYW4gVHVyaW5nIHByb3Bvc2VzIHRoZSAiVHVyaW5nIFRlc3QiIGZvciBtYWNoaW5lIGludGVsbGlnZW5jZS4KLSAqKjE5NTYqKjogRGFydG1vdXRoIENvbmZlcmVuY2UgbWFya3MgdGhlIGJpcnRoIG9mIEFJIGFzIGEgZmllbGQuCi0gKioxOTk3Kio6IElCTSdzIERlZXAgQmx1ZSBkZWZlYXRzIHdvcmxkIGNoZXNzIGNoYW1waW9uIEdhcnJ5IEthc3Bhcm92LgotICoqMjAxMSoqOiBJQk0gV2F0c29uIHdpbnMgSmVvcGFyZHkhIGFnYWluc3QgaHVtYW4gY2hhbXBpb25zLgotICoqMjAxNioqOiBHb29nbGUncyBBbHBoYUdvIGRlZmVhdHMgTGVlIFNlZG9sIGluIHRoZSBnYW1lIG9mIEdvLgotICoqMjAyMioqOiBDaGF0R1BUIGxhdW5jaGVzLCBkZW1vbnN0cmF0aW5nIGFkdmFuY2VkIGNvbnZlcnNhdGlvbmFsIEFJLgoKIyMgVHlwZXMgb2YgQUkKMS4gKipOYXJyb3cgQUkgKEFOSSkqKjogRGVzaWduZWQgZm9yIHNwZWNpZmljIHRhc2tzIChlLmcuLCBpbWFnZSByZWNvZ25pdGlvbiwgbGFuZ3VhZ2UgdHJhbnNsYXRpb24pLgoyLiAqKkdlbmVyYWwgQUkgKEFHSSkqKjogSHlwb3RoZXRpY2FsIEFJIHdpdGggaHVtYW4tbGV2ZWwgY29nbml0aXZlIGFiaWxpdGllcyBhY3Jvc3MgYWxsIGRvbWFpbnMuCjMuICoqU3VwZXJpbnRlbGxpZ2VudCBBSSAoQVNJKSoqOiBUaGVvcmV0aWNhbCBBSSBzdXJwYXNzaW5nIGh1bWFuIGludGVsbGlnZW5jZSBlbnRpcmVseS4KCiMjIEN1cnJlbnQgQXBwbGljYXRpb25zCi0gKipIZWFsdGhjYXJlKio6IE1lZGljYWwgaW1hZ2luZyBhbmFseXNpcywgZHJ1ZyBkaXNjb3ZlcnksIHBlcnNvbmFsaXplZCB0cmVhdG1lbnQgcGxhbnMuCi0gKipGaW5hbmNlKio6IEZyYXVkIGRldGVjdGlvbiwgYWxnb3JpdGhtaWMgdHJhZGluZywgY3JlZGl0IHNjb3JpbmcuCi0gKipUcmFuc3BvcnRhdGlvbioqOiBBdXRvbm9tb3VzIHZlaGljbGVzLCB0cmFmZmljIG9wdGltaXphdGlvbiwgcm91dGUgcGxhbm5pbmcuCi0gKipFZHVjYXRpb24qKjogUGVyc29uYWxpemVkIGxlYXJuaW5nLCBhdXRvbWF0ZWQgZ3JhZGluZywgdHV0b3Jpbmcgc3lzdGVtcy4KCiMjIEtleSBUZWNobm9sb2dpZXMKVGhlIG1vZGVybiBBSSByZXZvbHV0aW9uIGlzIGJ1aWx0IG9uOgotICoqRGVlcCBMZWFybmluZyoqOiBOZXVyYWwgbmV0d29ya3Mgd2l0aCBtdWx0aXBsZSBsYXllcnMgZm9yIGZlYXR1cmUgZXh0cmFjdGlvbi4KLSAqKlRyYW5zZm9ybWVycyoqOiBBcmNoaXRlY3R1cmUgYmVoaW5kIGxhcmdlIGxhbmd1YWdlIG1vZGVscyBsaWtlIEdQVC00IGFuZCBDbGF1ZGUuCi0gKipSZWluZm9yY2VtZW50IExlYXJuaW5nKio6IFRyYWluaW5nIGFnZW50cyB0aHJvdWdoIHJld2FyZC1iYXNlZCBmZWVkYmFjay4KCiMjIEV0aGljYWwgQ29uc2lkZXJhdGlvbnMKLSBCaWFzIGluIHRyYWluaW5nIGRhdGEgbGVhZGluZyB0byB1bmZhaXIgb3V0Y29tZXMuCi0gSm9iIGRpc3BsYWNlbWVudCBhbmQgZWNvbm9taWMgaW5lcXVhbGl0eS4KLSBQcml2YWN5IGNvbmNlcm5zIGZyb20gZGF0YSBjb2xsZWN0aW9uLgotIFBvdGVudGlhbCBtaXN1c2UgZm9yIG1pc2luZm9ybWF0aW9uIGFuZCBzdXJ2ZWlsbGFuY2UuCgojIyBUaGUgRnV0dXJlCkV4cGVydHMgcHJlZGljdCBBSSB3aWxsIGNvbnRpbnVlIHRvIHRyYW5zZm9ybSBpbmR1c3RyaWVzLCB3aXRoIGVzdGltYXRlZCBlY29ub21pYyBpbXBhY3QgCm9mICQxNS43IHRyaWxsaW9uIGJ5IDIwMzAgYWNjb3JkaW5nIHRvIFB3Qy4gVGhlIGRldmVsb3BtZW50IG9mIEFHSSByZW1haW5zIGEgbG9uZy10ZXJtIGdvYWwuCiIiIgoKRVNTQVlfU1BBQ0UgPSAiIiIKIyBFeHBsb3JpbmcgdGhlIFNvbGFyIFN5c3RlbQoKIyMgT3VyIENvc21pYyBOZWlnaGJvcmhvb2QKVGhlIFNvbGFyIFN5c3RlbSBjb25zaXN0cyBvZiB0aGUgU3VuLCBlaWdodCBwbGFuZXRzLCBkd2FyZiBwbGFuZXRzLCBtb29ucywgYXN0ZXJvaWRzLCAKYW5kIGNvbWV0cy4gSXQgZm9ybWVkIGFwcHJveGltYXRlbHkgNC42IGJpbGxpb24geWVhcnMgYWdvIGZyb20gYSBnaWFudCBtb2xlY3VsYXIgY2xvdWQuCgojIyBUaGUgU3VuCi0gKipUeXBlKio6IEctdHlwZSBtYWluLXNlcXVlbmNlIHN0YXIgKHllbGxvdyBkd2FyZikKLSAqKkRpYW1ldGVyKio6IDEsMzkyLDcwMCBrbSAoMTA5IHRpbWVzIEVhcnRoJ3MgZGlhbWV0ZXIpCi0gKipTdXJmYWNlIFRlbXBlcmF0dXJlKio6IDUsNTAwwrBDICg5LDkwMMKwRikKLSAqKkNvcmUgVGVtcGVyYXR1cmUqKjogMTUgbWlsbGlvbsKwQwotICoqRGlzdGFuY2UgZnJvbSBFYXJ0aCoqOiAxNDkuNiBtaWxsaW9uIGttICgxIEFzdHJvbm9taWNhbCBVbml0KQoKIyMgVGhlIFBsYW5ldHMKIyMjIElubmVyIFBsYW5ldHMgKFJvY2t5L1RlcnJlc3RyaWFsKQoxLiAqKk1lcmN1cnkqKjogU21hbGxlc3QgcGxhbmV0LCA4OC1kYXkgb3JiaXQsIG5vIG1vb25zLgoyLiAqKlZlbnVzKio6IEhvdHRlc3QgcGxhbmV0ICg0NzXCsEMgc3VyZmFjZSksIHJvdGF0ZXMgYmFja3dhcmRzLgozLiAqKkVhcnRoKio6IE9ubHkga25vd24gcGxhbmV0IHdpdGggbGlxdWlkIHdhdGVyIG9uIHN1cmZhY2UgYW5kIGxpZmUuCjQuICoqTWFycyoqOiAiUmVkIFBsYW5ldCIsIGhhcyB0aGUgbGFyZ2VzdCB2b2xjYW5vIChPbHltcHVzIE1vbnMsIDIxLjkga20gaGlnaCkuCgojIyMgT3V0ZXIgUGxhbmV0cyAoR2FzL0ljZSBHaWFudHMpCjUuICoqSnVwaXRlcioqOiBMYXJnZXN0IHBsYW5ldCAoMTF4IEVhcnRoJ3MgZGlhbWV0ZXIpLCBHcmVhdCBSZWQgU3BvdCBzdG9ybS4KNi4gKipTYXR1cm4qKjogRmFtb3VzIGZvciBpdHMgcmluZyBzeXN0ZW0sIDgzIGtub3duIG1vb25zIGluY2x1ZGluZyBUaXRhbi4KNy4gKipVcmFudXMqKjogUm90YXRlcyBvbiBpdHMgc2lkZSAoOTjCsCBheGlhbCB0aWx0KSwgMjcgbW9vbnMuCjguICoqTmVwdHVuZSoqOiBTdHJvbmdlc3Qgd2luZHMgaW4gdGhlIFNvbGFyIFN5c3RlbSAoMiwxMDAga20vaCkuCgojIyBOb3RhYmxlIE1pc3Npb25zCi0gKipWb3lhZ2VyIDEgJiAyICgxOTc3KSoqOiBGaXJzdCBzcGFjZWNyYWZ0IHRvIHJlYWNoIGludGVyc3RlbGxhciBzcGFjZS4KLSAqKkh1YmJsZSBTcGFjZSBUZWxlc2NvcGUgKDE5OTApKio6IFJldm9sdXRpb25pemVkIG91ciB2aWV3IG9mIHRoZSB1bml2ZXJzZS4KLSAqKk1hcnMgUm92ZXJzKio6IFNwaXJpdCwgT3Bwb3J0dW5pdHksIEN1cmlvc2l0eSwgUGVyc2V2ZXJhbmNlIGV4cGxvcmVkIE1hcnMuCi0gKipKYW1lcyBXZWJiIFNwYWNlIFRlbGVzY29wZSAoMjAyMSkqKjogTW9zdCBwb3dlcmZ1bCBzcGFjZSB0ZWxlc2NvcGUgZXZlciBidWlsdC4KCiMjIER3YXJmIFBsYW5ldHMKUGx1dG8gd2FzIHJlY2xhc3NpZmllZCBhcyBhIGR3YXJmIHBsYW5ldCBpbiAyMDA2IGJ5IHRoZSBJbnRlcm5hdGlvbmFsIEFzdHJvbm9taWNhbCBVbmlvbi4gCk90aGVyIGR3YXJmIHBsYW5ldHMgaW5jbHVkZSBFcmlzLCBIYXVtZWEsIE1ha2VtYWtlLCBhbmQgQ2VyZXMuCgojIyBGdXR1cmUgb2YgU3BhY2UgRXhwbG9yYXRpb24KLSAqKjIwMjQtMjAyNSoqOiBBcnRlbWlzIHByb2dyYW0gYWltcyB0byByZXR1cm4gaHVtYW5zIHRvIHRoZSBNb29uLgotICoqMjAzMHMqKjogTkFTQSBhbmQgU3BhY2VYIHBsYW5uaW5nIGNyZXdlZCBtaXNzaW9ucyB0byBNYXJzLgotICoqTG9uZy10ZXJtKio6IFBvdGVudGlhbCBjb2xvbml6YXRpb24gb2YgTWFycyBhbmQgYXN0ZXJvaWQgbWluaW5nLgoiIiIKCiMgUXVlc3Rpb25zIGZvciB0ZXN0aW5nICh3aXRoIGV4cGVjdGVkIGFuc3dlcnMgZnJvbSB0aGUgZXNzYXlzKQpURVNUX1FVRVNUSU9OUyA9IHsKICAgICJjbGltYXRlIjogWwogICAgICAgICgiV2hhdCBpcyB0aGUgY3VycmVudCBDTzIgY29uY2VudHJhdGlvbiBpbiB0aGUgYXRtb3NwaGVyZT8iLCAiNDIxIHBhcnRzIHBlciBtaWxsaW9uIChwcG0pIGluIDIwMjMiKSwKICAgICAgICAoIkhvdyBtdWNoIGhhcyB0aGUgZ2xvYmFsIHRlbXBlcmF0dXJlIHJpc2VuIHNpbmNlIHByZS1pbmR1c3RyaWFsIHRpbWVzPyIsICJhcHByb3hpbWF0ZWx5IDEuMcKwQyIpLAogICAgICAgICgiV2hhdCB5ZWFyIHdhcyB0aGUgUGFyaXMgQWdyZWVtZW50IHNpZ25lZD8iLCAiMjAxNSIpLAogICAgICAgICgiSG93IG11Y2ggaGF2ZSBzZWEgbGV2ZWxzIHJpc2VuIHNpbmNlIDE5MDA/IiwgImFib3V0IDIwIGNlbnRpbWV0ZXJzICg4IGluY2hlcykiKSwKICAgIF0sCiAgICAiYWkiOiBbCiAgICAgICAgKCJXaG8gY29pbmVkIHRoZSB0ZXJtICdBcnRpZmljaWFsIEludGVsbGlnZW5jZSc/IiwgIkpvaG4gTWNDYXJ0aHkgaW4gMTk1NiIpLAogICAgICAgICgiV2hlbiBkaWQgRGVlcCBCbHVlIGRlZmVhdCBHYXJyeSBLYXNwYXJvdj8iLCAiMTk5NyIpLAogICAgICAgICgiV2hhdCBpcyB0aGUgZXN0aW1hdGVkIGVjb25vbWljIGltcGFjdCBvZiBBSSBieSAyMDMwPyIsICIkMTUuNyB0cmlsbGlvbiBhY2NvcmRpbmcgdG8gUHdDIiksCiAgICAgICAgKCJXaGF0IGFyY2hpdGVjdHVyZSBwb3dlcnMgR1BULTQgYW5kIENsYXVkZT8iLCAiVHJhbnNmb3JtZXJzIiksCiAgICBdLAogICAgInNwYWNlIjogWwogICAgICAgICgiSG93IG9sZCBpcyB0aGUgU29sYXIgU3lzdGVtPyIsICJhcHByb3hpbWF0ZWx5IDQuNiBiaWxsaW9uIHllYXJzIiksCiAgICAgICAgKCJXaGF0IGlzIHRoZSB0YWxsZXN0IHZvbGNhbm8gaW4gdGhlIFNvbGFyIFN5c3RlbT8iLCAiT2x5bXB1cyBNb25zLCAyMS45IGttIGhpZ2giKSwKICAgICAgICAoIldoZW4gd2FzIHRoZSBKYW1lcyBXZWJiIFNwYWNlIFRlbGVzY29wZSBsYXVuY2hlZD8iLCAiMjAyMSIpLAogICAgICAgICgiV2hhdCB5ZWFyIHdhcyBQbHV0byByZWNsYXNzaWZpZWQgYXMgYSBkd2FyZiBwbGFuZXQ/IiwgIjIwMDYiKSwKICAgIF0sCn0KCmRlZiBnZXRfYWxsX2Vzc2F5cygpOgogICAgIiIiUmV0dXJuIGFsbCBlc3NheXMgYXMgYSBkaWN0aW9uYXJ5LiIiIgogICAgcmV0dXJuIHsKICAgICAgICAiY2xpbWF0ZSI6IEVTU0FZX0NMSU1BVEUsCiAgICAgICAgImFpIjogRVNTQVlfQUksCiAgICAgICAgInNwYWNlIjogRVNTQVlfU1BBQ0UsCiAgICB9CgpkZWYgZ2V0X3Rlc3RfcXVlc3Rpb25zKCk6CiAgICAiIiJSZXR1cm4gdGVzdCBxdWVzdGlvbnMgd2l0aCBleHBlY3RlZCBhbnN3ZXJzLiIiIgogICAgcmV0dXJuIFRFU1RfUVVFU1RJT05TCg==',
    'projects/hybrid_titans/essay_rag_demo.py': '...',
    'projects/hybrid_titans/hybrid_titan_demo.py': 'IiIiCkh5YnJpZCBUaXRhbiBEZW1vOiBUaXRhblJBRyBSZXRyaWV2YWwgKyBHUFQtMiBHZW5lcmF0aW9uCgpUaGlzIGV4cGVyaW1lbnQgZGVtb25zdHJhdGVzIERpcmVjdGlvbiBBIGZyb20gdGhlIHJlc2VhcmNoIHBsYW46Ci0gVGl0YW5SQUc6IFBlcmZvcm1zIHNlbWFudGljIHJldHJpZXZhbCB1c2luZyBNZW1vcnkgKyBFbnNlbWJsZSBGdXNpb24KLSBHUFQtMjogR2VuZXJhdGVzIG5hdHVyYWwgbGFuZ3VhZ2UgYW5zd2VycyBiYXNlZCBvbiByZXRyaWV2ZWQgY29udGV4dAoKQXJjaGl0ZWN0dXJlOgogICAgUXVlc3Rpb24g4oaSIFRpdGFuUkFHIOKGkiBDb250ZXh0IOKGkiBHUFQtMiDihpIgQW5zd2VyCiIiIgoKaW1wb3J0IHRvcmNoCmltcG9ydCB0b3JjaC5ubiBhcyBubgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IHN5cwppbXBvcnQgb3MKCiMgQWRkIHBhcmVudCBkaXJlY3RvcnkgdG8gcGF0aCAoaHlicmlkX3RpdGFucyAtPiBwcm9qZWN0cyAtPiBUaXRhbkxMTSkKc3lzLnBhdGguaW5zZXJ0KDAsIG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKSkpCgpmcm9tIG1haW4gaW1wb3J0IFRpdGFuUkFHCmZyb20gc2FtcGxlX2Vzc2F5cyBpbXBvcnQgRVNTQVlfQ0xJTUFURSwgRVNTQVlfQUksIEVTU0FZX1NQQUNFLCBURVNUX1FVRVNUSU9OUwoKCmNsYXNzIFNlbnRlbmNlVHJhbnNmb3JtZXJFbWJlZGRlcihubi5Nb2R1bGUpOgogICAgIiIiU2VtYW50aWMgZW1iZWRkZXIgdXNpbmcgc2VudGVuY2UtdHJhbnNmb3JtZXJzLiIiIgogICAgCiAgICBkZWYgX19pbml0X18oc2VsZiwgbW9kZWxfbmFtZT0iYWxsLU1pbmlMTS1MNi12MiIsIHRhcmdldF9kaW09MjU2KToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBmcm9tIHNlbnRlbmNlX3RyYW5zZm9ybWVycyBpbXBvcnQgU2VudGVuY2VUcmFuc2Zvcm1lcgogICAgICAgIHNlbGYubW9kZWwgPSBTZW50ZW5jZVRyYW5zZm9ybWVyKG1vZGVsX25hbWUpCiAgICAgICAgc2VsZi5zb3VyY2VfZGltID0gc2VsZi5tb2RlbC5nZXRfc2VudGVuY2VfZW1iZWRkaW5nX2RpbWVuc2lvbigpCiAgICAgICAgc2VsZi50YXJnZXRfZGltID0gdGFyZ2V0X2RpbQogICAgICAgIAogICAgICAgIGlmIHNlbGYuc291cmNlX2RpbSAhPSB0YXJnZXRfZGltOgogICAgICAgICAgICBzZWxmLnByb2plY3Rpb24gPSBubi5MaW5lYXIoc2VsZi5zb3VyY2VfZGltLCB0YXJnZXRfZGltKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlbGYucHJvamVjdGlvbiA9IE5vbmUKICAgICAgICAKICAgICAgICBwcmludChmIuKchSBMb2FkZWQgU2VudGVuY2VUcmFuc2Zvcm1lcjoge21vZGVsX25hbWV9IChkaW09e3NlbGYuc291cmNlX2RpbX0pIikKICAgIAogICAgZGVmIGZvcndhcmQoc2VsZiwgdGV4dCk6CiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgIGVtYmVkZGluZyA9IHNlbGYubW9kZWwuZW5jb2RlKHRleHQsIGNvbnZlcnRfdG9fdGVuc29yPVRydWUpCiAgICAgICAgICAgIGVtYmVkZGluZyA9IGVtYmVkZGluZy5jbG9uZSgpLmNwdSgpCiAgICAgICAgCiAgICAgICAgaWYgc2VsZi5wcm9qZWN0aW9uIGlzIG5vdCBOb25lOgogICAgICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgICAgIGVtYmVkZGluZyA9IHNlbGYucHJvamVjdGlvbihlbWJlZGRpbmcpCiAgICAgICAgCiAgICAgICAgaWYgZW1iZWRkaW5nLmRpbSgpID09IDE6CiAgICAgICAgICAgIGVtYmVkZGluZyA9IGVtYmVkZGluZy51bnNxdWVlemUoMCkKICAgICAgICByZXR1cm4gZW1iZWRkaW5nLnVuc3F1ZWV6ZSgwKQoKCmNsYXNzIEh5YnJpZFRpdGFuUkFHKG5uLk1vZHVsZSk6CiAgICAiIiIKICAgIEh5YnJpZCBUaXRhbjogVGl0YW5SQUcgUmV0cmlldmFsICsgTExNIEdlbmVyYXRpb24KICAgIAogICAgVGhpcyBjb21iaW5lczoKICAgIDEuIFRpdGFuUkFHIGZvciBzZW1hbnRpYyBtZW1vcnktYmFzZWQgcmV0cmlldmFsCiAgICAyLiBFeHRlcm5hbCBMTE0gKEdQVC0yKSBmb3IgYW5zd2VyIGdlbmVyYXRpb24KICAgICIiIgogICAgCiAgICBkZWYgX19pbml0X18oc2VsZiwgdGl0YW5fcmFnLCBlbWJlZGRlciwgbGxtX25hbWU9Imdvb2dsZS9mbGFuLXQ1LWxhcmdlIik6CiAgICAgICAgc3VwZXIoKS5fX2luaXRfXygpCiAgICAgICAgc2VsZi50aXRhbl9yYWcgPSB0aXRhbl9yYWcKICAgICAgICBzZWxmLmVtYmVkZGVyID0gZW1iZWRkZXIKICAgICAgICAKICAgICAgICAjIExvYWQgRmxhbi1UNSBmb3IgZ2VuZXJhdGlvbiAoYmV0dGVyIGluc3RydWN0aW9uIGZvbGxvd2luZyB0aGFuIEdQVC0yKQogICAgICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBUNUZvckNvbmRpdGlvbmFsR2VuZXJhdGlvbiwgVDVUb2tlbml6ZXIKICAgICAgICBzZWxmLnRva2VuaXplciA9IFQ1VG9rZW5pemVyLmZyb21fcHJldHJhaW5lZChsbG1fbmFtZSkKICAgICAgICBzZWxmLmxsbSA9IFQ1Rm9yQ29uZGl0aW9uYWxHZW5lcmF0aW9uLmZyb21fcHJldHJhaW5lZChsbG1fbmFtZSkKICAgICAgICAKICAgICAgICBwcmludChmIuKchSBMb2FkZWQgTExNOiB7bGxtX25hbWV9IikKICAgIAogICAgZGVmIHJldHJpZXZlKHNlbGYsIHF1ZXN0aW9uLCBlc3NheV9saW5lcywgbGluZV9lbWJlZGRpbmdzLCB0b3BrPTUpOgogICAgICAgICIiIgogICAgICAgIFJldHJpZXZlIHJlbGV2YW50IGNvbnRleHQgdXNpbmcgRW5zZW1ibGUgRnVzaW9uLgogICAgICAgIChDb21iaW5lcyBLZXl3b3JkICsgTWVtb3J5ICsgRW1iZWRkaW5nIGZvciByb2J1c3QgcmV0cmlldmFsKQogICAgICAgIAogICAgICAgIFJldHVybnM6CiAgICAgICAgICAgIGNvbnRleHQ6IFJldHJpZXZlZCB0ZXh0CiAgICAgICAgICAgIGRldGFpbHM6IERpY3Qgd2l0aCByZXRyaWV2YWwgc2NvcmVzCiAgICAgICAgIiIiCiAgICAgICAgZGltID0gc2VsZi5lbWJlZGRlci50YXJnZXRfZGltCiAgICAgICAgCiAgICAgICAgIyBFbWJlZCBxdWVzdGlvbgogICAgICAgIHF1ZXJ5X2VtYiA9IHNlbGYuZW1iZWRkZXIocXVlc3Rpb24pCiAgICAgICAgcXVlcnlfZmxhdCA9IHF1ZXJ5X2VtYi5yZXNoYXBlKC0xLCBkaW0pCiAgICAgICAgCiAgICAgICAgd2l0aCB0b3JjaC5ub19ncmFkKCk6CiAgICAgICAgICAgICMgTWV0aG9kIDE6IEtleXdvcmQgc2NvcmVzCiAgICAgICAgICAgIHF1ZXN0aW9uX2xvd2VyID0gcXVlc3Rpb24ubG93ZXIoKQogICAgICAgICAgICBrZXl3b3JkX3Njb3JlcyA9IFtdCiAgICAgICAgICAgIGZvciBsaW5lIGluIGVzc2F5X2xpbmVzOgogICAgICAgICAgICAgICAgc2NvcmUgPSBzdW0oMSBmb3Igd29yZCBpbiBxdWVzdGlvbl9sb3dlci5zcGxpdCgpIAogICAgICAgICAgICAgICAgICAgICAgICAgICBpZiBsZW4od29yZCkgPiAzIGFuZCB3b3JkIGluIGxpbmUubG93ZXIoKSkKICAgICAgICAgICAgICAgIGtleXdvcmRfc2NvcmVzLmFwcGVuZChmbG9hdChzY29yZSkpCiAgICAgICAgICAgIGtleXdvcmRfc2NvcmVzID0gdG9yY2gudGVuc29yKGtleXdvcmRfc2NvcmVzKQogICAgICAgICAgICAKICAgICAgICAgICAgIyBNZXRob2QgMjogTWVtb3J5IHNjb3JlcyAoTFRNIHNlbWFudGljKQogICAgICAgICAgICBtZW1vcnlfb3V0cHV0ID0gc2VsZi50aXRhbl9yYWcudGl0YW4ubHRtLmZvcndhcmRfbm9fdXBkYXRlKHF1ZXJ5X2ZsYXQpCiAgICAgICAgICAgIG1lbW9yeV92ZWMgPSBtZW1vcnlfb3V0cHV0Lm1lYW4oZGltPTApCiAgICAgICAgICAgIG1lbW9yeV9zY29yZXMgPSB0b3JjaC5ubi5mdW5jdGlvbmFsLmNvc2luZV9zaW1pbGFyaXR5KAogICAgICAgICAgICAgICAgbWVtb3J5X3ZlYy51bnNxdWVlemUoMCksIGxpbmVfZW1iZWRkaW5ncwogICAgICAgICAgICApCiAgICAgICAgICAgIAogICAgICAgICAgICAjIE1ldGhvZCAzOiBEaXJlY3QgZW1iZWRkaW5nIHNpbWlsYXJpdHkKICAgICAgICAgICAgcXVlcnlfdmVjID0gcXVlcnlfZmxhdC5tZWFuKGRpbT0wKQogICAgICAgICAgICBlbWJlZGRpbmdfc2NvcmVzID0gdG9yY2gubm4uZnVuY3Rpb25hbC5jb3NpbmVfc2ltaWxhcml0eSgKICAgICAgICAgICAgICAgIHF1ZXJ5X3ZlYy51bnNxdWVlemUoMCksIGxpbmVfZW1iZWRkaW5ncwogICAgICAgICAgICApCiAgICAgICAgICAgIAogICAgICAgICAgICAjIE5vcm1hbGl6ZQogICAgICAgICAgICBkZWYgbm9ybWFsaXplKHNjb3Jlcyk6CiAgICAgICAgICAgICAgICBtaW5fcywgbWF4X3MgPSBzY29yZXMubWluKCksIHNjb3Jlcy5tYXgoKQogICAgICAgICAgICAgICAgaWYgbWF4X3MgLSBtaW5fcyA8IDFlLTY6CiAgICAgICAgICAgICAgICAgICAgcmV0dXJuIHRvcmNoLnplcm9zX2xpa2Uoc2NvcmVzKQogICAgICAgICAgICAgICAgcmV0dXJuIChzY29yZXMgLSBtaW5fcykgLyAobWF4X3MgLSBtaW5fcykKICAgICAgICAgICAgCiAgICAgICAgICAgICMgRW5zZW1ibGUgRnVzaW9uIChiYWxhbmNlZCB3ZWlnaHRzKQogICAgICAgICAgICBmdXNlZF9zY29yZXMgPSAoCiAgICAgICAgICAgICAgICAwLjIgKiBub3JtYWxpemUoa2V5d29yZF9zY29yZXMpICsKICAgICAgICAgICAgICAgIDAuNSAqIG5vcm1hbGl6ZShtZW1vcnlfc2NvcmVzKSArCiAgICAgICAgICAgICAgICAwLjMgKiBub3JtYWxpemUoZW1iZWRkaW5nX3Njb3JlcykKICAgICAgICAgICAgKQogICAgICAgICAgICAKICAgICAgICAgICAgIyBHZXQgdG9wLWsgaW5kaWNlcwogICAgICAgICAgICB0b3BrX3ZhbHVlcywgdG9wa19pbmRpY2VzID0gZnVzZWRfc2NvcmVzLnRvcGsobWluKHRvcGssIGxlbihlc3NheV9saW5lcykpKQogICAgICAgICAgICAKICAgICAgICAgICAgIyBCdWlsZCBjb250ZXh0IGZyb20gdG9wLWsgbGluZXMKICAgICAgICAgICAgY29udGV4dF9saW5lcyA9IFtlc3NheV9saW5lc1tpZHhdIGZvciBpZHggaW4gdG9wa19pbmRpY2VzLnRvbGlzdCgpXQogICAgICAgICAgICBjb250ZXh0ID0gIiAiLmpvaW4oY29udGV4dF9saW5lcykKICAgICAgICAKICAgICAgICByZXR1cm4gY29udGV4dCwgewogICAgICAgICAgICAia2V5d29yZF90b3AxIjoga2V5d29yZF9zY29yZXMuYXJnbWF4KCkuaXRlbSgpLAogICAgICAgICAgICAibWVtb3J5X3RvcDEiOiBtZW1vcnlfc2NvcmVzLmFyZ21heCgpLml0ZW0oKSwKICAgICAgICAgICAgImVtYmVkX3RvcDEiOiBlbWJlZGRpbmdfc2NvcmVzLmFyZ21heCgpLml0ZW0oKSwKICAgICAgICAgICAgImZ1c2VkX3RvcDEiOiBmdXNlZF9zY29yZXMuYXJnbWF4KCkuaXRlbSgpLAogICAgICAgICAgICAidG9wa19pbmRpY2VzIjogdG9wa19pbmRpY2VzLnRvbGlzdCgpCiAgICAgICAgfQogICAgCiAgICBkZWYgZ2VuZXJhdGUoc2VsZiwgcXVlc3Rpb24sIGNvbnRleHQsIG1heF9uZXdfdG9rZW5zPTEwMCk6CiAgICAgICAgIiIiCiAgICAgICAgR2VuZXJhdGUgYW5zd2VyIHVzaW5nIEZsYW4tVDUgd2l0aCByZXRyaWV2ZWQgY29udGV4dC4KICAgICAgICAiIiIKICAgICAgICAjIEJ1aWxkIHByb21wdCBvcHRpbWl6ZWQgZm9yIEZsYW4tVDUgaW5zdHJ1Y3Rpb24gZm9ybWF0CiAgICAgICAgcHJvbXB0ID0gZiIiIkJhc2VkIG9uIHRoZSBmb2xsb3dpbmcgY29udGV4dCwgYW5zd2VyIHRoZSBxdWVzdGlvbiB3aXRoIHNwZWNpZmljIGZhY3RzIGFuZCBudW1iZXJzLgoKQ29udGV4dDoge2NvbnRleHR9CgpRdWVzdGlvbjoge3F1ZXN0aW9ufQoKQW5zd2VyOiIiIgogICAgICAgIAogICAgICAgICMgVG9rZW5pemUKICAgICAgICBpbnB1dHMgPSBzZWxmLnRva2VuaXplcigKICAgICAgICAgICAgcHJvbXB0LCAKICAgICAgICAgICAgcmV0dXJuX3RlbnNvcnM9InB0IiwgCiAgICAgICAgICAgIHRydW5jYXRpb249VHJ1ZSwgCiAgICAgICAgICAgIG1heF9sZW5ndGg9NTEyCiAgICAgICAgKQogICAgICAgIAogICAgICAgICMgR2VuZXJhdGUgKGdyZWVkeSBkZWNvZGluZyBmb3IgbW9yZSBhY2N1cmF0ZSBleHRyYWN0aW9uKQogICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICBvdXRwdXRzID0gc2VsZi5sbG0uZ2VuZXJhdGUoCiAgICAgICAgICAgICAgICAqKmlucHV0cywKICAgICAgICAgICAgICAgIG1heF9uZXdfdG9rZW5zPW1heF9uZXdfdG9rZW5zLAogICAgICAgICAgICAgICAgbnVtX2JlYW1zPTMsICAjIEJlYW0gc2VhcmNoIGZvciBiZXR0ZXIgcXVhbGl0eQogICAgICAgICAgICAgICAgZWFybHlfc3RvcHBpbmc9VHJ1ZQogICAgICAgICAgICApCiAgICAgICAgCiAgICAgICAgIyBEZWNvZGUKICAgICAgICBhbnN3ZXIgPSBzZWxmLnRva2VuaXplci5kZWNvZGUob3V0cHV0c1swXSwgc2tpcF9zcGVjaWFsX3Rva2Vucz1UcnVlKQogICAgICAgIAogICAgICAgIHJldHVybiBhbnN3ZXIsIHByb21wdAogICAgCiAgICBkZWYgYW5zd2VyKHNlbGYsIHF1ZXN0aW9uLCBlc3NheV9saW5lcywgbGluZV9lbWJlZGRpbmdzLCBleHBlY3RlZF9hbnN3ZXI9Tm9uZSwgdG9waz01LCBtYXhfbmV3X3Rva2Vucz0xMDApOgogICAgICAgICIiIgogICAgICAgIEZ1bGwgcGlwZWxpbmU6IFJldHJpZXZlIGNvbnRleHQsIGdlbmVyYXRlIGFuc3dlciwgYW5kIGxlYXJuIGZyb20gUSZBLgogICAgICAgIAogICAgICAgIEFyZ3M6CiAgICAgICAgICAgIHF1ZXN0aW9uOiBUaGUgcXVlc3Rpb24gdG8gYW5zd2VyCiAgICAgICAgICAgIGVzc2F5X2xpbmVzOiBMaXN0IG9mIGVzc2F5IGxpbmVzCiAgICAgICAgICAgIGxpbmVfZW1iZWRkaW5nczogVGVuc29yIG9mIGxpbmUgZW1iZWRkaW5ncwogICAgICAgICAgICBleHBlY3RlZF9hbnN3ZXI6IElmIHByb3ZpZGVkLCBtb2RlbCBsZWFybnMgZnJvbSB0aGlzIFEmQSBwYWlyIChPbmxpbmUgTGVhcm5pbmcpCiAgICAgICAgICAgIHRvcGs6IE51bWJlciBvZiBjb250ZXh0IGxpbmVzIHRvIHJldHJpZXZlCiAgICAgICAgICAgIG1heF9uZXdfdG9rZW5zOiBNYXggdG9rZW5zIHRvIGdlbmVyYXRlCiAgICAgICAgIiIiCiAgICAgICAgZGltID0gc2VsZi5lbWJlZGRlci50YXJnZXRfZGltCiAgICAgICAgCiAgICAgICAgIyBFbWJlZCBxdWVzdGlvbgogICAgICAgIHF1ZXJ5X2VtYiA9IHNlbGYuZW1iZWRkZXIocXVlc3Rpb24pCiAgICAgICAgcXVlcnlfZmxhdCA9IHF1ZXJ5X2VtYi5yZXNoYXBlKC0xLCBkaW0pCiAgICAgICAgCiAgICAgICAgIyBTdGVwIDE6IFJldHJpZXZlCiAgICAgICAgY29udGV4dCwgcmV0cmlldmFsX2RldGFpbHMgPSBzZWxmLnJldHJpZXZlKAogICAgICAgICAgICBxdWVzdGlvbiwgZXNzYXlfbGluZXMsIGxpbmVfZW1iZWRkaW5ncywgdG9wawogICAgICAgICkKICAgICAgICAKICAgICAgICAjIFN0ZXAgMjogR2VuZXJhdGUKICAgICAgICBhbnN3ZXIsIHByb21wdCA9IHNlbGYuZ2VuZXJhdGUocXVlc3Rpb24sIGNvbnRleHQsIG1heF9uZXdfdG9rZW5zKQogICAgICAgIAogICAgICAgICMgU3RlcCAzOiBPbmxpbmUgTGVhcm5pbmcgKFRydWUgVFRUISkgLSBpZiBleHBlY3RlZCBhbnN3ZXIgcHJvdmlkZWQKICAgICAgICBsZWFybmVkID0gRmFsc2UKICAgICAgICBpZiBleHBlY3RlZF9hbnN3ZXIgaXMgbm90IE5vbmU6CiAgICAgICAgICAgIGV4cGVjdGVkX2VtYiA9IHNlbGYuZW1iZWRkZXIoZXhwZWN0ZWRfYW5zd2VyKQogICAgICAgICAgICBleHBlY3RlZF92ZWMgPSBleHBlY3RlZF9lbWIucmVzaGFwZSgtMSwgZGltKS5tZWFuKGRpbT0wKQogICAgICAgICAgICAKICAgICAgICAgICAgIyBVcGRhdGUgbWVtb3J5OiBRdWVzdGlvbiDihpIgRXhwZWN0ZWQgQW5zd2VyCiAgICAgICAgICAgIHNlbGYudGl0YW5fcmFnLnRpdGFuLmx0bS5mb3J3YXJkX3dpdGhfdXBkYXRlKAogICAgICAgICAgICAgICAgcXVlcnlfZmxhdC5tZWFuKGRpbT0wLCBrZWVwZGltPVRydWUpLCAgIyBrZXkgPSBxdWVzdGlvbgogICAgICAgICAgICAgICAgZXhwZWN0ZWRfdmVjLnVuc3F1ZWV6ZSgwKSAgICAgICAgICAgICAgIyB2YWx1ZSA9IGFuc3dlcgogICAgICAgICAgICApCiAgICAgICAgICAgIGxlYXJuZWQgPSBUcnVlCiAgICAgICAgCiAgICAgICAgcmV0dXJuIHsKICAgICAgICAgICAgImFuc3dlciI6IGFuc3dlciwKICAgICAgICAgICAgImNvbnRleHQiOiBjb250ZXh0LAogICAgICAgICAgICAicmV0cmlldmFsIjogcmV0cmlldmFsX2RldGFpbHMsCiAgICAgICAgICAgICJwcm9tcHQiOiBwcm9tcHQsCiAgICAgICAgICAgICJvbmxpbmVfbGVhcm5lZCI6IGxlYXJuZWQKICAgICAgICB9CgoKZGVmIHJ1bl9oeWJyaWRfdGl0YW5fZGVtbyhhcmdzKToKICAgICIiIlJ1biB0aGUgSHlicmlkIFRpdGFuIGRlbW8uIiIiCiAgICAKICAgIHByaW50KCI9IiAqIDYwKQogICAgcHJpbnQoIkhZQlJJRCBUSVRBTiBERU1PIikKICAgIHByaW50KCJUaXRhblJBRyBSZXRyaWV2YWwgKyBHUFQtMiBHZW5lcmF0aW9uIikKICAgIHByaW50KCI9IiAqIDYwKQogICAgCiAgICAjIFNlbGVjdCBlc3NheSBiYXNlZCBvbiBhcmd1bWVudAogICAgZXNzYXlfa2V5ID0gYXJncy5lc3NheS5sb3dlcigpCiAgICBpZiBlc3NheV9rZXkgPT0gImNsaW1hdGUiOgogICAgICAgIGVzc2F5X3RleHQgPSBFU1NBWV9DTElNQVRFCiAgICAgICAgZXNzYXlfcXVlc3Rpb25zID0gVEVTVF9RVUVTVElPTlNbImNsaW1hdGUiXQogICAgZWxpZiBlc3NheV9rZXkgPT0gImFpIjoKICAgICAgICBlc3NheV90ZXh0ID0gRVNTQVlfQUkKICAgICAgICBlc3NheV9xdWVzdGlvbnMgPSBURVNUX1FVRVNUSU9OU1siYWkiXQogICAgZWxpZiBlc3NheV9rZXkgPT0gInNwYWNlIjoKICAgICAgICBlc3NheV90ZXh0ID0gRVNTQVlfU1BBQ0UKICAgICAgICBlc3NheV9xdWVzdGlvbnMgPSBURVNUX1FVRVNUSU9OU1sic3BhY2UiXQogICAgZWxzZToKICAgICAgICBlc3NheV90ZXh0ID0gRVNTQVlfQ0xJTUFURQogICAgICAgIGVzc2F5X3F1ZXN0aW9ucyA9IFRFU1RfUVVFU1RJT05TWyJjbGltYXRlIl0KICAgIAogICAgcHJpbnQoZiJcbvCfk5YgU2VsZWN0ZWQgRXNzYXk6IHtlc3NheV9rZXkudXBwZXIoKX0iKQogICAgcHJpbnQoZiIgICBMZW5ndGg6IHtsZW4oZXNzYXlfdGV4dCl9IGNoYXJhY3RlcnMiKQogICAgCiAgICAjIEluaXRpYWxpemUgZW1iZWRkZXIKICAgIGVtYmVkZGVyID0gU2VudGVuY2VUcmFuc2Zvcm1lckVtYmVkZGVyKHRhcmdldF9kaW09YXJncy5kaW0pCiAgICAKICAgICMgSW5pdGlhbGl6ZSBUaXRhblJBRyAoZmlyc3QgY3JlYXRlIGJhc2UgbW9kZWwsIHRoZW4gd3JhcCkKICAgIGZyb20gbWFpbiBpbXBvcnQgVGl0YW5NQUcKICAgIAogICAgYmFzZV9tb2RlbCA9IFRpdGFuTUFHKAogICAgICAgIGRpbT1hcmdzLmRpbSwgCiAgICAgICAgd2luZG93X3NpemU9MzIsIAogICAgICAgIGhpZGRlbl9kaW09YXJncy5kaW0gKiAyLAogICAgICAgIG1lbW9yeV9kZXB0aD0zLCAKICAgICAgICBudW1fcGVyc2lzdGVudF90b2tlbnM9NCwgCiAgICAgICAgdGhyZXNob2xkPTAuMAogICAgKQogICAgCiAgICAjIENvbmZpZ3VyZSBmb3IgaGlnaC1maWRlbGl0eSBtZW1vcnkKICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIGJhc2VfbW9kZWwubHRtLnRoZXRhLmZpbGxfKDAuMDEpICAgIyBMZWFybmluZyByYXRlCiAgICAgICAgYmFzZV9tb2RlbC5sdG0uYWxwaGEuZmlsbF8oMC4wMDAxKSAjIE1pbmltYWwgZm9yZ2V0dGluZwogICAgCiAgICB0aXRhbl9yYWcgPSBUaXRhblJBRyhiYXNlX21vZGVsKQogICAgCiAgICAjIFBhcnNlIGVzc2F5IGludG8gUEFSQUdSQVBIUyAobm90IGxpbmVzKSBmb3IgYmV0dGVyIGNvbnRleHQgcmV0ZW50aW9uCiAgICAjIFNwbGl0IGJ5IGRvdWJsZSBuZXdsaW5lcywgdGhlbiBjbGVhbiB1cCBlYWNoIHBhcmFncmFwaAogICAgcmF3X3BhcmFncmFwaHMgPSBlc3NheV90ZXh0LnNwbGl0KCdcblxuJykKICAgIGVzc2F5X3BhcmFncmFwaHMgPSBbXQogICAgZm9yIHBhcmEgaW4gcmF3X3BhcmFncmFwaHM6CiAgICAgICAgIyBNZXJnZSBsaW5lcyB3aXRoaW4gcGFyYWdyYXBoLCBjbGVhbiB1cAogICAgICAgIGNsZWFuZWQgPSAnICcuam9pbihsaW5lLnN0cmlwKCkgZm9yIGxpbmUgaW4gcGFyYS5zcGxpdCgnXG4nKSBpZiBsaW5lLnN0cmlwKCkpCiAgICAgICAgaWYgY2xlYW5lZCBhbmQgbGVuKGNsZWFuZWQpID4gMTA6ICAjIFNraXAgZW1wdHkgb3IgdmVyeSBzaG9ydAogICAgICAgICAgICBlc3NheV9wYXJhZ3JhcGhzLmFwcGVuZChjbGVhbmVkKQogICAgCiAgICBwcmludChmIlxu8J+UhCBFbWJlZGRpbmcge2xlbihlc3NheV9wYXJhZ3JhcGhzKX0gcGFyYWdyYXBocy4uLiIpCiAgICAKICAgICMgRW1iZWQgYWxsIHBhcmFncmFwaHMKICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgIHBhcmFfZW1iZWRkaW5ncyA9IFtdCiAgICAgICAgZm9yIHBhcmEgaW4gZXNzYXlfcGFyYWdyYXBoczoKICAgICAgICAgICAgcGFyYV9lbWIgPSBlbWJlZGRlcihwYXJhKQogICAgICAgICAgICBwYXJhX3ZlYyA9IHBhcmFfZW1iLnJlc2hhcGUoLTEsIGFyZ3MuZGltKS5tZWFuKGRpbT0wKQogICAgICAgICAgICBwYXJhX2VtYmVkZGluZ3MuYXBwZW5kKHBhcmFfdmVjKQogICAgICAgIGxpbmVfZW1iZWRkaW5ncyA9IHRvcmNoLnN0YWNrKHBhcmFfZW1iZWRkaW5ncykgICMgS2VlcCB2YXJpYWJsZSBuYW1lIGZvciBjb21wYXRpYmlsaXR5CiAgICAKICAgICMgVXNlIHBhcmFncmFwaHMgaW5zdGVhZCBvZiBsaW5lcwogICAgZXNzYXlfbGluZXMgPSBlc3NheV9wYXJhZ3JhcGhzICAjIEFsaWFzIGZvciBjb21wYXRpYmlsaXR5CiAgICAKICAgICMgRGlnZXN0IGludG8gVGl0YW5SQUcKICAgIHByaW50KGYiXG7wn6egIERpZ2VzdGluZyBpbnRvIFRpdGFuUkFHICh7YXJncy5lcG9jaHN9IGVwb2NocykuLi4iKQogICAgYWxsX2VtYmVkZGluZ3MgPSB0b3JjaC5jYXQoW2VtYmVkZGVyKGxpbmUpIGZvciBsaW5lIGluIGVzc2F5X2xpbmVzXSwgZGltPTEpCiAgICAKICAgIGZvciBlcG9jaCBpbiByYW5nZShhcmdzLmVwb2Nocyk6CiAgICAgICAgdGl0YW5fcmFnLmRpZ2VzdF9rbm93bGVkZ2UoYWxsX2VtYmVkZGluZ3MpCiAgICAgICAgaWYgKGVwb2NoICsgMSkgJSAxMCA9PSAwOgogICAgICAgICAgICBwcmludChmIiAgIEVwb2NoIHtlcG9jaCArIDF9L3thcmdzLmVwb2Noc30gY29tcGxldGVkIikKICAgIAogICAgIyBJbml0aWFsaXplIEh5YnJpZCBUaXRhbiB3aXRoIEZsYW4tVDUKICAgIHByaW50KCJcbvCfpJYgSW5pdGlhbGl6aW5nIEh5YnJpZCBUaXRhbiAoRmxhbi1UNSBnZW5lcmF0b3IpLi4uIikKICAgIGh5YnJpZF90aXRhbiA9IEh5YnJpZFRpdGFuUkFHKHRpdGFuX3JhZywgZW1iZWRkZXIpICAjIFVzZXMgZGVmYXVsdCBGbGFuLVQ1CiAgICAKICAgICMgUnVuIFEmQSB3aXRoIE9ubGluZSBMZWFybmluZwogICAgcHJpbnQoIlxuIiArICI9IiAqIDYwKQogICAgcHJpbnQoIlFVRVNUSU9OIEFOU1dFUklORyB3aXRoIE9OTElORSBMRUFSTklORyIpCiAgICBwcmludCgiPSIgKiA2MCkKICAgIAogICAgZm9yIGksIChxdWVzdGlvbiwgZXhwZWN0ZWQpIGluIGVudW1lcmF0ZShlc3NheV9xdWVzdGlvbnMsIDEpOgogICAgICAgIHByaW50KGYiXG57J+KUgCcgKiA2MH0iKQogICAgICAgIHByaW50KGYiUXtpfToge3F1ZXN0aW9ufSIpCiAgICAgICAgcHJpbnQoZiJFeHBlY3RlZDoge2V4cGVjdGVkfSIpCiAgICAgICAgCiAgICAgICAgIyBHZXQgYW5zd2VyIGZyb20gSHlicmlkIFRpdGFuICh3aXRoIE9ubGluZSBMZWFybmluZyEpCiAgICAgICAgcmVzdWx0ID0gaHlicmlkX3RpdGFuLmFuc3dlcigKICAgICAgICAgICAgcXVlc3Rpb24sIAogICAgICAgICAgICBlc3NheV9saW5lcywgCiAgICAgICAgICAgIGxpbmVfZW1iZWRkaW5ncywKICAgICAgICAgICAgZXhwZWN0ZWRfYW5zd2VyPWV4cGVjdGVkLCAgIyBFbmFibGUgT25saW5lIExlYXJuaW5nCiAgICAgICAgICAgIHRvcGs9YXJncy50b3BrLAogICAgICAgICAgICBtYXhfbmV3X3Rva2Vucz1hcmdzLm1heF90b2tlbnMKICAgICAgICApCiAgICAgICAgCiAgICAgICAgcHJpbnQoZiJcbltSZXRyaWV2YWxdIEluZGljZXM6IHtyZXN1bHRbJ3JldHJpZXZhbCddWyd0b3BrX2luZGljZXMnXX0iKQogICAgICAgIHByaW50KGYiW0NvbnRleHRdXG4gIHtyZXN1bHRbJ2NvbnRleHQnXX0iKQogICAgICAgIHByaW50KGYiXG5bR2VuZXJhdGVkIEFuc3dlcl0iKQogICAgICAgIHByaW50KGYiICB7cmVzdWx0WydhbnN3ZXInXX0iKQogICAgICAgIAogICAgICAgICMgU2hvdyBPbmxpbmUgTGVhcm5pbmcgc3RhdHVzCiAgICAgICAgaWYgcmVzdWx0LmdldCgnb25saW5lX2xlYXJuZWQnKToKICAgICAgICAgICAgcHJpbnQoIiAg8J+TmiBPbmxpbmUgTGVhcm5pbmc6IFVwZGF0ZWQgbWVtb3J5IHdpdGggdGhpcyBRJkEiKQogICAgICAgIAogICAgICAgICMgU2ltcGxlIGV2YWx1YXRpb246IGNoZWNrIGlmIGV4cGVjdGVkIGFuc3dlciBrZXl3b3JkcyBhcHBlYXIKICAgICAgICBleHBlY3RlZF93b3JkcyA9IGV4cGVjdGVkLmxvd2VyKCkuc3BsaXQoKQogICAgICAgIGFuc3dlcl9sb3dlciA9IHJlc3VsdFsnYW5zd2VyJ10ubG93ZXIoKQogICAgICAgIG1hdGNoZXMgPSBzdW0oMSBmb3Igd29yZCBpbiBleHBlY3RlZF93b3JkcyBpZiBsZW4od29yZCkgPiAzIGFuZCB3b3JkIGluIGFuc3dlcl9sb3dlcikKICAgICAgICBtYXRjaF9yYXRpbyA9IG1hdGNoZXMgLyBsZW4oW3cgZm9yIHcgaW4gZXhwZWN0ZWRfd29yZHMgaWYgbGVuKHcpID4gM10pIGlmIGV4cGVjdGVkX3dvcmRzIGVsc2UgMAogICAgICAgIAogICAgICAgIGlmIG1hdGNoX3JhdGlvID4gMC4zOgogICAgICAgICAgICBwcmludChmIiAg4pyFIENvbnRhaW5zIGV4cGVjdGVkIGtleXdvcmRzICh7bWF0Y2hfcmF0aW86LjAlfSkiKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHByaW50KGYiICDimqDvuI8gTWlzc2luZyBleHBlY3RlZCBrZXl3b3JkcyAoe21hdGNoX3JhdGlvOi4wJX0pIikKICAgIAogICAgcHJpbnQoIlxuIiArICI9IiAqIDYwKQogICAgcHJpbnQoIuKchSBIeWJyaWQgVGl0YW4gRGVtbyBDb21wbGV0ZSEiKQogICAgcHJpbnQoIj0iICogNjApCiAgICAKICAgIHJldHVybiBoeWJyaWRfdGl0YW4KCgpkZWYgbWFpbigpOgogICAgcGFyc2VyID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoZGVzY3JpcHRpb249Ikh5YnJpZCBUaXRhbiBEZW1vIikKICAgIHBhcnNlci5hZGRfYXJndW1lbnQoIi0tZXNzYXkiLCB0eXBlPXN0ciwgZGVmYXVsdD0iY2xpbWF0ZSIsCiAgICAgICAgICAgICAgICAgICAgICAgY2hvaWNlcz1bImNsaW1hdGUiLCAiYWkiLCAic3BhY2UiXSwKICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJFc3NheSB0byB1c2UiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS1kaW0iLCB0eXBlPWludCwgZGVmYXVsdD0yNTYsCiAgICAgICAgICAgICAgICAgICAgICAgaGVscD0iRW1iZWRkaW5nIGRpbWVuc2lvbiIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLWVwb2NocyIsIHR5cGU9aW50LCBkZWZhdWx0PTUwLAogICAgICAgICAgICAgICAgICAgICAgIGhlbHA9IkRpZ2VzdGlvbiBlcG9jaHMiKQogICAgcGFyc2VyLmFkZF9hcmd1bWVudCgiLS10b3BrIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MywKICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJUb3AtayBsaW5lcyB0byByZXRyaWV2ZSIpCiAgICBwYXJzZXIuYWRkX2FyZ3VtZW50KCItLW1heF90b2tlbnMiLCB0eXBlPWludCwgZGVmYXVsdD01MCwKICAgICAgICAgICAgICAgICAgICAgICBoZWxwPSJNYXggdG9rZW5zIHRvIGdlbmVyYXRlIikKICAgIAogICAgYXJncyA9IHBhcnNlci5wYXJzZV9hcmdzKCkKICAgIHJ1bl9oeWJyaWRfdGl0YW5fZGVtbyhhcmdzKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK'
}

for path, content in files.items():
    write_file(path, content)

print('Files in current directory:', os.listdir('.'))
if os.path.exists('experiments'):
    print('Files in experiments:', os.listdir('experiments'))


Current working directory: /content
Created /content/main.py
Created /content/projects/original_benchmarks/associative_recall.py
Created /content/projects/original_benchmarks/train_toy.py
Created /content/projects/hybrid_titans/rag_compare.py
Created /content/projects/hybrid_titans/sample_essays.py
Created /content/projects/hybrid_titans/essay_rag_demo.py
Created /content/projects/hybrid_titans/hybrid_titan_demo.py
Files in current directory: ['.config', 'projects', 'main.py', 'sample_data']


In [ ]:
# Run Baseline (Dense Update)
# Decreasing vocab_size to 16 to ensure model can learn quickly
import os
if os.path.exists('projects/original_benchmarks/train_toy.py'):
    print('Running Baseline (Dense Update, Vocab=16)...')
    !python projects/original_benchmarks/train_toy.py --model_variant MAC --num_epochs 5 --batch_size 16 --vocab_size 16 --threshold 0.0
else:
    print('ERROR: projects/original_benchmarks/train_toy.py not found!')


In [ ]:
# Run Innovation (Sparse Update)
print('Running Sparse Update (Threshold = 0.5, Vocab=16)...')
!python projects/original_benchmarks/train_toy.py --model_variant MAC --num_epochs 5 --batch_size 16 --vocab_size 16 --threshold 0.5


In [ ]:
# TitanRAG Prototype Demo
# Demonstrating 'Retrieve-then-Digest' mechanism
import torch
from main import TitanMAC, TitanRAG

print('Initializing TitanRAG...')
# 1. Initialize Base Model
base_model = TitanMAC(dim=128, chunk_size=32, hidden_dim=128, memory_depth=2, num_persistent_tokens=4, threshold=0.1)
rag_model = TitanRAG(base_model)

# 2. Simulate Data (Embeddings)
print('Simulating Retrieval...')
doc1 = torch.randn(1, 64, 128) # Retrieved Doc A
doc2 = torch.randn(1, 64, 128) # Retrieved Doc B
query = torch.randn(1, 16, 128) # User Query

# 3. Run Pipeline
print('Executing Digestion & Inference...')
output = rag_model.query_with_context(query, [doc1, doc2])

print('Success! Output shape:', output.shape)
print('The model weights have been updated on-the-fly with document info.')


In [ ]:
# Experiment: RAG Comparison - Small Scale (Within Context Window)
# Both methods should succeed.
print('--- Running RAG Comparison (Small Scale: 10 Docs) ---')
!python projects/hybrid_titans/rag_compare.py --num_docs 10 --doc_len 32


In [ ]:
# Experiment: RAG Comparison - Large Scale (Exceeds Context Window)
# Traditional RAG should FAIL (Context Limit).
# TitanRAG should PASS (Infinite Memory).
print('\n--- Running RAG Comparison (Large Scale: 100 Docs) ---')
!python projects/hybrid_titans/rag_compare.py --num_docs 100 --doc_len 32


## Essay-based TitanRAG Demo
This cell demonstrates TitanRAG on real text: Climate Essay.
The model digests the essay and then answers verification questions.

In [ ]:
# Run Essay RAG Demo (Climate Essay - Random Embedder Baseline)
print('--- Essay RAG Demo: Climate (Random Embedder) ---')
!python projects/hybrid_titans/essay_rag_demo.py --essay climate --epochs 50 --embedder random


## Semantic Embedder (sentence-transformers)
Now running with **semantic embeddings** for meaningful text similarity.
This should show significant improvement in Hybrid RAG performance!

In [ ]:
# Run ALL Essays with Semantic Embedder
print('=' * 60)
print('SEMANTIC EMBEDDER TEST - ALL ESSAYS')
print('=' * 60)

# Climate Essay
print('\n--- Essay 1/3: Climate Change ---')
!python projects/hybrid_titans/essay_rag_demo.py --essay climate --epochs 50 --embedder semantic

# AI Essay
print('\n--- Essay 2/3: Artificial Intelligence ---')
!python projects/hybrid_titans/essay_rag_demo.py --essay ai --epochs 50 --embedder semantic

# Space Essay
print('\n--- Essay 3/3: Solar System ---')
!python projects/hybrid_titans/essay_rag_demo.py --essay space --epochs 50 --embedder semantic


## Hybrid Titan: TitanRAG + Flan-T5 Generation\n
This cell demonstrates **Direction A** from the research plan:\n
- TitanRAG performs retrieval using Ensemble Fusion (Keyword + Memory + Embedding)\n
- Flan-T5 generates natural language answers from context\n
- **Online Learning**: Model learns from each Q&A pair

In [3]:
# Hybrid Titan Demo: TitanRAG Retrieval + Flan-T5 Generation
# Running ALL THREE ESSAYS with Online Learning

print('=' * 60)
print('HYBRID TITAN DEMO - ALL ESSAYS')
print('=' * 60)

# 1. Climate Essay
print('\n' + '─' * 60)
print('📖 Essay 1/3: CLIMATE CHANGE')
print('─' * 60)
!python projects/hybrid_titans/hybrid_titan_demo.py --essay climate --epochs 50 --topk 5

# 2. AI Essay
print('\n' + '─' * 60)
print('📖 Essay 2/3: ARTIFICIAL INTELLIGENCE')
print('─' * 60)
!python projects/hybrid_titans/hybrid_titan_demo.py --essay ai --epochs 50 --topk 5

# 3. Space Essay
print('\n' + '─' * 60)
print('📖 Essay 3/3: SOLAR SYSTEM')
print('─' * 60)
!python projects/hybrid_titans/hybrid_titan_demo.py --essay space --epochs 50 --topk 5


HYBRID TITAN DEMO - ALL ESSAYS

────────────────────────────────────────────────────────────
📖 Essay 1/3: CLIMATE CHANGE
────────────────────────────────────────────────────────────
2026-01-24 02:50:19.049403: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769223019.068640    3138 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769223019.074474    3138 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769223019.089302    3138 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769223019.089323    3138 computation_placer.cc:177] computation pla